# Rainfall and Traffic Flow Across Eight Cities

How does traffic flow vary with hourly rainfall across eight cities, and what patterns appear after rain ends?

The analysis compares observed flow with matched dry conditions and follows qualifying rain events using a fixed panel of sensors. These comparisons describe associations. Cloud results are provisional because the merged export lacks the information needed to verify the original weather join.

The verified cloud run covered **23,799 city-hours** and **118 qualifying rain events**. Of the 41 events with a measured flow deficit, 23 had confirmed recoveries.

To explore the results in Colab, run the notebook through **Run and validate the analysis**. Then run **My Web App** and **Open the dashboard**. The public CSV is about 1.92 GB, so allow time for the download and analysis.

## Dataset and provenance

The Integrated Urban Traffic-Flood (IUTF) dataset combines traffic sensor records with hourly weather. This analysis uses city, sensor ID, timestamp, mean flow, and precipitation. Speed and occupancy coverage vary by city.

The verified Cloud Storage object was generation `1789228531819061` with MD5 `b0fece30ea500230072c785891535b6e`. The reference dataset describes 40 cities. The historical merged CSV had records for 38, and the loader reports the coverage of the current file each time it runs.

- [Public Google Cloud Storage CSV](https://storage.googleapis.com/bucket-quickstart-iutfdataset/all_cities_combined.csv)
- [Original notebook that links the CSV](https://github.com/divyakapoor23/IUTF_Viz/blob/main/Project1.ipynb)
- [IUTF reference dataset, version 1](https://doi.org/10.6084/m9.figshare.30022807.v1)
- [IUTF dataset paper](https://www.nature.com/articles/s41597-025-06336-3)

### Measures at a glance

| Measure | Definition |
| --- | --- |
| Matched dry conditions | Rainfall below 0.1 mm/hour for the same city, sensor, hour of day, and weekday or weekend group |
| Well-observed rain event | A continuous wet spell with a peak of at least 0.5 mm/hour, dry hours on either side, and enough sensors to follow throughout |
| Confirmed recovery | Two consecutive dry hours within ±10% of baseline after an observed flow deficit of at least 10% during rain |

Sections 3 and 4 specify the minimum observation counts, sensor coverage, and follow-up rules.

### City selection

The final comparison covers Paris, Luzern, Hamburg, Toronto, Essen, Innsbruck, Darmstadt, and Manchester. Selection prioritized observed days, repeated rain exposures, usable event windows, and dry-baseline support. Manchester contributes measured speed to the source audit but has only 28 observed days and weaker dry-baseline support.

The historical tables compare this selection with the original list, which also included London, Madrid, Graz, and Santander.

### Unresolved weather join

The original export joined weather timestamps with the UTC timezone removed to traffic timestamps and filled unmatched precipitation with zero. The Cloud Storage CSV omits the raw weather timestamps and match status. As a result, some apparent dry hours may be failed matches.

The analysis converts precipitation to mm/hour and rebuilds the rainfall classes and dry baselines. Cloud mode uses the merged timestamps as supplied. The optional raw-Parquet workflow applies an interval correction using the original weather records.

In [64]:
# @title Install notebook dependencies
%pip -q install 'pandas>=2,<4' 'plotly>=5.24,<6.0' 'dash>=2.11' dash-bootstrap-components

Note: you may need to restart the kernel to use updated packages.


## Historical data audit

The embedded tables summarize the earlier local audit of 10,489,938 rows from 38 cities. Tokyo and Stuttgart lacked local hourly reading files. In the merged CSV, millimetre thresholds were applied to precipitation stored in metres, placing all rows in the same rainfall category.

The earlier `iutf_project.csv` contained London, Madrid, and Melbourne. The audit used the supplied local weather timestamps and a preliminary dry/light threshold below 0.5 mm/hour. Its event-window counts help explain city selection. Sections 3 and 4 govern the current analysis.

Strasbourg was a close alternate. The city selection reflects judgement about data coverage and quality.

In [65]:
# @title Historical audit data (embedded snapshot)
from io import StringIO
import pandas as pd

# Embedded results, not a fresh source read.
audit_profile = pd.read_csv(StringIO('city,status,rows,sensors,observed_days,observed_hours,start,end,span_days,duplicate_sensor_hours,speed_available_pct,flow_missing_pct,negative_flow_rows,zero_flow_pct,duplicate_detector_ids,unmatched_detector_pct,coordinates_missing_pct,duplicate_weather_grid_hours,negative_rain_rows,weather_matched_pct,old_utc_unmatched_pct,rain_changed_by_time_alignment_hours,moderate_hours,heavy_hours,extreme_hours,wet_hours,dry_light_hours,max_rain_mm,observed_wet_onsets,complete_recovery_windows,wet_sensor_rows,wet_speed_available_pct,wet_baseline_missing_pct,wet_baseline_under3_pct\naugsburg,profiled,271144.0,713.0,20.0,480.0,2017-05-06 00:00:00,2017-05-25 23:00:00,20.0,0.0,0.0,0.0,0.0,9.853804620423096,0.0,0.0,0.0,0.0,0.0,100.0,0.4166666666666667,281.0,34.0,0.0,0.0,34.0,446.0,2.3381710052490234,10.0,5.0,23435.0,0.0,8.052058886281204,100.0\nbasel,profiled,13013.0,77.0,8.0,169.0,2016-10-24 00:00:00,2016-10-31 00:00:00,8.0,0.0,0.0,0.0,0.0,13.793898409283026,0.0,0.0,0.0,0.0,0.0,100.0,0.591715976331361,77.0,10.0,0.0,0.0,10.0,159.0,2.4166107177734375,3.0,1.0,770.0,0.0,100.0,100.0\nbern,profiled,127283.0,755.0,8.0,169.0,2016-10-24 00:00:00,2016-10-31 00:00:00,8.0,0.0,0.0,0.0,0.0,2.1212573556562933,0.0,0.0,0.0,0.0,0.0,100.0,0.591715976331361,85.0,24.0,0.0,0.0,24.0,145.0,1.6255378723144531,4.0,2.0,18120.0,0.0,100.0,100.0\nbirmingham,profiled,10332.0,52.0,12.0,227.0,2017-10-23 07:00:00,2017-11-18 11:00:00,27.0,0.0,96.10917537746806,0.0,0.0,3.8908246225319396,0.0,0.0,0.0,0.0,0.0,100.0,0.0,46.0,0.0,0.0,0.0,0.0,227.0,0.3581047058105469,0.0,0.0,0.0,,,\nbolton,profiled,20750.0,166.0,6.0,125.0,2017-11-13 08:00:00,2017-11-18 12:00:00,6.0,0.0,100.0,0.0,0.0,19.937349397590364,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,125.0,0.148773193359375,0.0,0.0,0.0,,,\nbordeaux,profiled,70368.0,425.0,7.0,168.0,2016-11-21 00:00:00,2016-11-27 23:00:00,7.0,0.0,0.0,0.0,0.0,0.7773419736243747,0.0,0.0,0.0,0.0,0.0,100.0,0.5952380952380952,95.0,21.0,0.0,0.0,21.0,147.0,2.5534629821777344,6.0,2.0,8806.0,0.0,100.0,100.0\nbremen,profiled,178972.0,548.0,14.0,336.0,2016-09-19 00:00:00,2016-10-02 23:00:00,14.0,0.0,0.0,0.0,0.0,1.2158326442125025,0.0,0.0,0.0,0.0,0.0,100.0,0.5952380952380952,116.0,3.0,0.0,0.0,3.0,333.0,2.1402835845947266,2.0,2.0,1606.0,0.0,0.809464508094645,100.0\ncagliari,profiled,142125.0,122.0,50.0,1200.0,2016-05-16 00:00:00,2016-07-29 23:00:00,75.0,0.0,0.0,0.0,0.0,1.4543535620052772,0.0,0.0,0.0,0.0,0.0,100.0,1.6666666666666667,90.0,0.0,0.0,0.0,0.0,1200.0,0.396728515625,0.0,0.0,0.0,,,\nconstance,profiled,20496.0,122.0,7.0,168.0,2017-02-13 00:00:00,2017-02-19 23:00:00,7.0,0.0,100.0,0.0,0.0,6.552498048399688,0.0,0.0,0.0,0.0,0.0,100.0,0.5952380952380952,48.0,8.0,0.0,0.0,8.0,160.0,1.2106895446777344,1.0,1.0,976.0,100.0,100.0,100.0\ndarmstadt,profiled,266951.0,392.0,39.0,900.0,2015-09-21 00:00:00,2016-05-06 23:00:00,229.0,0.0,0.0,0.0,0.0,13.492363767133295,0.0,0.0,0.0,0.0,0.0,100.0,2.0,460.0,41.0,0.0,0.0,41.0,859.0,1.5299320220947266,14.0,5.0,12176.0,0.0,0.21353482260183967,3.818988173455979\nessen,profiled,29663.0,36.0,35.0,824.0,2017-03-27 10:00:00,2017-09-30 23:00:00,188.0,0.0,100.0,0.0,0.0,0.07079526683073188,0.0,0.0,0.0,0.0,0.0,100.0,0.48543689320388345,523.0,58.0,1.0,0.0,59.0,765.0,4.2629241943359375,19.0,10.0,2124.0,100.0,0.0,0.0\nfrankfurt,profiled,2688.0,112.0,1.0,24.0,2016-12-21 00:00:00,2016-12-21 23:00:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,4.166666666666666,5.0,0.0,0.0,0.0,0.0,24.0,0.0019073486328125,0.0,0.0,0.0,,,\ngraz,profiled,70920.0,300.0,10.0,240.0,2016-04-04 00:00:00,2016-09-23 23:00:00,173.0,0.0,0.0,0.0,0.0,10.400451212633953,0.0,0.0,0.0,0.0,0.0,100.0,1.6666666666666667,96.0,12.0,0.0,0.0,12.0,228.0,2.231597900390625,2.0,2.0,3492.0,0.0,0.0,100.0\ngroningen,profiled,2420.0,55.0,2.0,44.0,2017-09-21 00:00:00,2017-10-06 21:00:00,16.0,0.0,100.0,0.0,0.0,7.727272727272727,0.0,0.0,0.0,0.0,0.0,100.0,0.0,23.0,2.0,0.0,0.0,2.0,42.0,2.4535655975341797,2.0,0.0,110.0,100.0,100.0,100.0\nhamburg,profiled,1022272.0,418.0,105.0,2508.0,2016-08-27 00:00:00,2016-12-09 11:00:00,105.0,0.0,0.0,0.0,0.0,2.2178050460151506,0.0,0.0,0.0,0.0,0.0,100.0,0.03987240829346093,1222.0,57.0,0.0,0.0,57.0,2451.0,3.292083740234375,24.0,21.0,23160.0,0.0,0.0,0.0\ninnsbruck,profiled,11520.0,16.0,30.0,720.0,2017-04-01 00:00:00,2017-04-30 23:00:00,30.0,0.0,100.0,0.0,0.0,10.972222222222221,0.0,0.0,0.0,0.0,0.0,100.0,0.2777777777777778,543.0,119.0,0.0,0.0,119.0,601.0,2.0384788513183594,29.0,9.0,1904.0,100.0,0.0,27.73109243697479\nkassel,profiled,48259.0,505.0,6.0,99.0,2016-08-28 23:00:00,2016-09-02 01:00:00,6.0,0.0,0.0,0.0,0.0,2.8264157980894757,0.0,0.0,0.0,0.0,0.0,100.0,0.0,29.0,0.0,0.0,0.0,0.0,99.0,0.148773193359375,0.0,0.0,0.0,,,\nlondon,profiled,2886733.0,5719.0,23.0,542.0,2015-05-15 00:00:00,2016-05-22 23:00:00,374.0,0.0,0.0,0.0,0.0,16.54863820103903,0.0,0.0,0.0,0.0,0.0,95.5719557195572,4.981549815498155,266.0,15.0,0.0,0.0,15.0,503.0,1.660744349161784,7.0,1.0,80162.0,0.0,0.4678026995334448,100.0\nlosangeles,profiled,233279.0,1722.0,6.0,144.0,2017-10-02 00:00:00,2017-10-10 23:00:00,9.0,0.0,0.0,0.0,31.0,0.0034293699818672067,0.0,0.0,0.0,0.0,0.0,100.0,14.583333333333334,17.0,0.0,0.0,0.0,0.0,144.0,0.07677078247070312,0.0,0.0,0.0,,,\nluzern,profiled,1344504.0,158.0,366.0,8759.0,2015-01-01 00:00:00,2016-01-01 00:00:00,366.0,0.0,0.0,0.0,0.0,5.864988129451456,0.0,0.0,0.0,0.0,0.0,99.71457928987327,0.29683753853179584,5041.0,1074.0,23.0,0.0,1097.0,7637.0,6.211757659912109,237.0,85.0,167326.0,0.0,0.0,0.0\nmadrid,profiled,427094.0,1116.0,20.0,384.0,2016-08-29 00:00:00,2017-11-11 09:00:00,440.0,0.0,0.0,0.0,105.0,2.069801963970461,0.0,0.0,0.0,0.0,0.0,100.0,0.78125,169.0,28.0,0.0,0.0,28.0,356.0,3.1604766845703125,6.0,2.0,31248.0,0.0,0.33922171018945213,78.64823348694317\nmanchester,profiled,96199.0,181.0,28.0,588.0,2017-09-08 08:00:00,2017-11-18 12:00:00,72.0,0.0,100.0,0.0,0.0,7.4979989396979185,0.0,0.0,0.0,0.0,0.0,100.0,0.0,312.0,20.0,0.0,0.0,20.0,568.0,2.7360916137695312,12.0,8.0,3196.0,100.0,0.0,59.94993742177722\nmarseille,profiled,121849.0,169.0,31.0,721.0,2017-06-01 00:00:00,2017-07-01 00:00:00,31.0,0.0,0.0,0.0,3981.0,0.007386191105384534,0.0,0.0,0.0,0.0,0.0,100.0,0.0,112.0,2.0,0.0,0.0,2.0,719.0,0.8237361907958984,2.0,2.0,338.0,0.0,0.0,0.0\nmelbourne,profiled,342614.0,954.0,15.0,360.0,2018-02-12 00:00:00,2018-02-26 23:00:00,15.0,0.0,0.0,0.0,0.0,0.38819195946458696,0.0,0.0,0.0,0.0,0.0,100.0,3.0555555555555554,121.0,9.0,0.0,0.0,9.0,351.0,2.47955322265625,2.0,2.0,8563.0,0.0,0.36202265561135116,100.0\nmunich,profiled,12700.0,520.0,2.0,25.0,2017-02-14 00:00:00,2017-02-15 00:00:00,2.0,0.0,0.0,0.0,0.0,0.7007874015748031,0.0,0.0,0.0,0.0,0.0,96.0,8.0,0.0,0.0,0.0,0.0,0.0,24.0,0.0,0.0,0.0,0.0,,,\nparis,profiled,1548314.0,208.0,336.0,8036.0,2016-01-01 01:00:00,2016-12-01 00:00:00,336.0,0.0,0.0,0.0,0.0,0.06510307340759044,0.0,0.0,0.0,0.0,0.0,99.70134395221503,0.2986560477849677,4722.0,400.0,2.0,0.0,402.0,7610.0,4.948139190673828,136.0,94.0,77588.0,0.0,0.0,0.0463989276692272\nrotterdam,profiled,29819.0,259.0,6.0,116.0,2017-09-21 00:00:00,2017-11-02 14:00:00,43.0,0.0,100.0,0.0,0.0,4.634628927864784,0.0,0.0,0.0,0.0,0.0,100.0,0.0,62.0,0.0,0.0,0.0,0.0,116.0,0.37789344787597656,0.0,0.0,0.0,,,\nsantander,profiled,41391.0,219.0,11.0,189.0,2016-06-17 14:00:00,2016-12-02 12:00:00,169.0,0.0,0.0,0.0,0.0,1.1886642023628324,0.0,0.0,0.0,0.0,0.0,100.0,1.0582010582010581,89.0,3.0,2.0,0.0,5.0,184.0,6.343841552734375,2.0,1.0,1095.0,0.0,100.0,100.0\nspeyer,profiled,61772.0,184.0,14.0,336.0,2016-09-19 00:00:00,2016-10-02 23:00:00,14.0,0.0,0.0,0.0,0.0,2.57560059573917,0.0,0.0,0.0,0.0,0.0,100.0,0.5952380952380952,122.0,4.0,0.0,0.0,4.0,332.0,0.8087158203125,3.0,3.0,736.0,0.0,0.0,100.0\nstrasbourg,profiled,67024.0,142.0,25.0,472.0,2017-05-10 14:00:00,2017-11-11 09:00:00,186.0,0.0,0.0,0.0,0.0,4.589400811649559,0.0,0.0,0.0,0.0,0.0,100.0,0.423728813559322,296.0,31.0,0.0,0.0,31.0,441.0,3.325939178466797,13.0,8.0,4402.0,0.0,0.0,58.06451612903226\nstuttgart,no hourly readings,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,\ntaipeh,profiled,121884.0,445.0,14.0,331.0,2017-09-18 00:00:00,2017-10-01 23:00:00,14.0,0.0,0.0,0.0,0.0,1.2938531718683337,0.0,0.0,0.0,0.0,0.0,100.0,2.416918429003021,182.0,8.0,2.0,0.0,10.0,321.0,4.851341247558594,6.0,4.0,3548.0,0.0,23.75986471251409,100.0\ntokyo,no hourly readings,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,\ntorino,profiled,177508.0,399.0,21.0,504.0,2016-09-26 00:00:00,2016-10-16 23:00:00,21.0,0.0,100.0,0.0,0.0,0.844468981679699,0.0,0.0,0.0,0.0,0.0,100.0,0.3968253968253968,315.0,63.0,0.0,0.0,63.0,441.0,3.259897232055664,12.0,4.0,21820.0,100.0,2.296058661778185,100.0\ntoronto,profiled,260680.0,188.0,61.0,1464.0,2016-09-01 00:00:00,2017-01-31 23:00:00,153.0,0.0,0.0,0.0,0.0,0.07365352155899954,0.0,0.0,0.0,0.0,0.0,100.0,0.6147540983606558,508.0,46.0,1.0,0.0,47.0,1417.0,6.389617919921875,22.0,10.0,8365.0,0.0,0.0,0.1434548714883443\ntoulouse,profiled,80136.0,477.0,7.0,168.0,2008-05-16 00:00:00,2008-06-27 23:00:00,43.0,0.0,0.0,0.0,1470.0,0.4142956973145652,1.0,0.0,,0.0,0.0,100.0,8.333333333333332,104.0,10.0,0.0,0.0,10.0,158.0,3.942251205444336,5.0,5.0,4770.0,0.0,0.0,0.0\nutrecht,profiled,97891.0,1027.0,4.0,96.0,2017-06-12 00:00:00,2017-06-15 23:00:00,4.0,0.0,0.0,0.0,196.0,5.413163620761868,0.0,0.0,0.0,0.0,0.0,100.0,2.083333333333333,40.0,0.0,0.0,0.0,0.0,96.0,0.18405914306640625,0.0,0.0,0.0,,,\nvilnius,profiled,13213.0,581.0,2.0,25.0,2015-03-17 00:00:00,2015-03-18 00:00:00,2.0,0.0,0.0,0.0,0.0,0.045409823658518124,0.0,0.0,0.0,0.0,0.0,96.0,12.0,0.0,0.0,0.0,0.0,0.0,24.0,0.0,0.0,0.0,0.0,,,\nwolfsburg,profiled,44688.0,133.0,14.0,336.0,2016-09-19 00:00:00,2016-10-02 23:00:00,14.0,0.0,0.0,0.0,0.0,6.211958467597565,0.0,0.0,0.0,0.0,0.0,100.0,0.5952380952380952,152.0,11.0,0.0,0.0,11.0,325.0,1.5535354614257812,3.0,3.0,1463.0,0.0,0.0,100.0\nzurich,profiled,171302.0,1020.0,7.0,168.0,2015-10-26 00:00:00,2015-11-01 23:00:00,7.0,0.0,0.0,0.0,0.0,0.5767591738566976,0.0,0.0,0.0,0.0,0.0,100.0,0.5952380952380952,82.0,8.0,0.0,0.0,8.0,160.0,1.039743423461914,2.0,0.0,8160.0,0.0,100.0,100.0\n'))
combined_profile = pd.read_csv(StringIO('city,rows,speed_missing,flow_missing,baseline_missing,zero_baseline,infinite_pct_change,coordinates_missing,non_light_category_rows,max_precipitation,observed_hours,observed_days\naugsburg,271144,271144,0,0,24154,0,0,0,0.002338171005249,480,20\nbasel,13013,13013,0,0,1785,0,0,0,0.0024166107177734,169,8\nbern,127283,127283,0,0,2524,0,0,0,0.0016255378723144,169,8\nbirmingham,10332,402,0,0,163,0,0,0,0.0003581047058105,227,12\nbolton,20750,0,0,0,4137,0,0,0,0.0001487731933593,125,6\nbordeaux,70368,70368,0,0,547,0,0,0,0.0025534629821777,168,7\nbremen,178972,178972,0,0,1305,0,0,0,0.0021402835845947,336,14\ncagliari,142125,142125,0,0,578,0,0,0,0.000396728515625,1200,50\nconstance,20496,0,0,0,1343,0,0,0,0.0012106895446777,168,7\ndarmstadt,266951,266951,0,0,19195,0,0,0,0.0015299320220947,900,39\nessen,29663,0,0,0,0,0,0,0,0.0042629241943359,824,35\nfrankfurt,2688,2688,0,0,0,0,0,0,1.9073486328125e-06,24,1\ngraz,70920,70920,0,0,2860,0,0,0,0.0022315979003906,240,10\ngroningen,2420,0,0,0,187,0,0,0,0.0024535655975341,44,2\nhamburg,1022272,1022272,0,0,35,0,0,0,0.0032920837402343,2508,105\ninnsbruck,11520,0,0,0,0,0,0,0,0.0020384788513183,720,30\nkassel,48259,48259,0,0,1364,0,0,0,0.0001487731933593,99,6\nlondon,2886733,2886733,0,0,195490,0,0,0,0.0016607443491617,542,23\nlosangeles,233279,233279,0,0,8,0,0,0,7.677078247070312e-05,144,6\nluzern,1344504,1344504,0,0,58905,0,0,0,0.0062117576599121,8759,366\nmadrid,427094,427094,0,0,1161,0,0,0,0.0031604766845703,384,20\nmanchester,96199,0,0,0,5976,0,0,0,0.0027360916137695,588,28\nmarseille,121849,121849,0,0,0,0,0,0,0.0008237361907958,721,31\nmelbourne,342614,342614,0,0,101,0,0,0,0.0024795532226562,360,15\nmunich,12700,12700,0,0,89,0,0,0,0.0,25,2\nparis,1548314,1548314,0,0,0,0,0,0,0.0049481391906738,8036,336\nrotterdam,29819,0,0,0,1321,0,0,0,0.0003778934478759,116,6\nsantander,41391,41391,0,0,204,0,0,0,0.0063438415527343,189,11\nspeyer,61772,61772,0,0,1244,0,0,0,0.0008087158203125,336,14\nstrasbourg,67024,67024,0,0,2088,0,0,0,0.0033259391784667,472,25\ntaipeh,121884,121884,0,0,1001,0,0,0,0.0048513412475585,331,14\ntorino,177508,0,0,0,815,0,0,0,0.0032598972320556,504,21\ntoronto,260680,260680,0,0,0,0,0,0,0.0063896179199218,1464,61\ntoulouse,80304,80304,0,0,14,0,0,0,0.0039422512054443,168,7\nutrecht,97891,97891,0,0,5299,0,0,0,0.0001840591430664,96,4\nvilnius,13213,13213,0,0,6,0,0,0,0.0,25,2\nwolfsburg,44688,44688,0,0,2370,0,0,0,0.0015535354614257,336,14\nzurich,171302,171302,0,0,988,0,0,0,0.0010397434234619,168,7\n'))
original_subset_profile = pd.read_csv(StringIO('city,rows,speed_missing,flow_missing,baseline_missing,zero_baseline,infinite_pct_change,coordinates_missing,non_light_category_rows,max_precipitation,observed_hours,observed_days\nlondon,2886733,2886733,0,0,195490,0,0,0,0.0016607443491617,542,23\nmadrid,427094,427094,0,0,1161,0,0,0,0.0031604766845703,384,20\nmelbourne,342614,342614,0,0,101,0,0,0,0.0024795532226562,360,15\n'))
historical_selected_cities = ['paris','luzern','hamburg','toronto','essen','innsbruck','darmstadt','manchester']
audit_profile.set_index('city').loc[historical_selected_cities, ['sensors','observed_days','observed_hours','speed_available_pct','wet_hours','complete_recovery_windows']].round(2)

,sensors,observed_days,observed_hours,speed_available_pct,wet_hours,complete_recovery_windows
city,,,,,,
paris,208.0,336.0,8036.0,0.0,402.0,94.0
luzern,158.0,366.0,8759.0,0.0,1097.0,85.0
hamburg,418.0,105.0,2508.0,0.0,57.0,21.0
toronto,188.0,61.0,1464.0,0.0,47.0,10.0
essen,36.0,35.0,824.0,100.0,59.0,10.0
innsbruck,16.0,30.0,720.0,100.0,119.0,9.0
darmstadt,392.0,39.0,900.0,0.0,41.0,5.0
manchester,181.0,28.0,588.0,100.0,20.0,8.0


### Compare the original eight cities

The table below shows observed days, speed availability, wet hours, and complete event windows for the original selection. Use it alongside the revised selection above to see why calendar span alone was a poor coverage measure.

In [66]:
original_cities = ['london','paris','madrid','santander','luzern','graz','essen','toronto']
audit_profile.set_index('city').loc[original_cities, ['observed_days','span_days','speed_available_pct','complete_recovery_windows']]

,observed_days,span_days,speed_available_pct,complete_recovery_windows
city,,,,
london,23.0,374.0,0.0,1.0
paris,336.0,336.0,0.0,94.0
madrid,20.0,440.0,0.0,2.0
santander,11.0,169.0,0.0,1.0
luzern,366.0,366.0,0.0,85.0
graz,10.0,173.0,0.0,2.0
essen,35.0,188.0,100.0,10.0
toronto,61.0,153.0,0.0,10.0


In [67]:
assert len(combined_profile) == 38
assert combined_profile.rows.sum() == 10489938
assert combined_profile.non_light_category_rows.sum() == 0
assert audit_profile.duplicate_sensor_hours.fillna(0).sum() == 0
assert combined_profile.rows.sum() - audit_profile.rows.sum() == 168
original_subset_profile[['city','rows','observed_days','speed_missing']]

,city,rows,observed_days,speed_missing
0,london,2886733,23,2886733
1,madrid,427094,20,427094
2,melbourne,342614,15,342614


### Findings from the local audit

- Essen, Innsbruck, and Manchester have measured speed in the revised eight-city selection. Innsbruck lacks occupancy. Flow is the primary measure across all cities.
- The original merged CSV contained 337,257 rows with a zero baseline. The analysis rebuilds baselines and requires a positive reference before computing percentage change.
- A duplicate detector metadata ID added 168 rows to the Toulouse merge. Toulouse is outside the selected eight, and raw city–sensor–hour keys were unique across the 38 available hourly sources.
- Observed dates ranged from 28 in Manchester to 366 in Luzern among the selected cities. London had 23 observed days and Madrid had 20 despite their long calendar spans.

## 1. Configure and load the data

Run **Install notebook dependencies**, **Setup and configuration**, and **Load selected city data from Google Cloud**. The default `DATA_SOURCE = 'google_cloud'` streams the public CSV and retains the eight selected cities. The loader reports the source row counts and each selected city's observed hours and dates.

For the optional raw-weather analysis, set `DATA_SOURCE = 'raw_parquet'`, place `IUTFD/` under `ROOT`, and use a Python environment with a Parquet reader. Cloud and raw runs write charts to separate output directories.

In [68]:
# @title Setup and configuration
import threading
from pathlib import Path
from urllib.request import urlopen

import dash_bootstrap_components as dbc
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from dash import Dash, Input, Output, dcc, html
from plotly.subplots import make_subplots
from IPython.display import display

try:
    import google.colab
    pio.renderers.default = 'colab'
except ImportError:
    pio.renderers.default = 'plotly_mimetype+notebook_connected'

try:
    pio.templates.default = 'plotly_white'
except Exception as e:
    print(f"Warning: Could not set Plotly default template due to: {e}")

# Outputs are local to the notebook runtime; cloud mode needs no local input folders.
ROOT = Path.cwd()
DATA_SOURCE = 'google_cloud'  # optional alternative: 'raw_parquet'
DATA_URL = 'https://storage.googleapis.com/bucket-quickstart-iutfdataset/all_cities_combined.csv'
METRES_TO_MM = 1000
DRY_THRESHOLD_MM = 0.1
EVENT_PEAK_MIN_MM = 0.5
OUT = ROOT / ('rainfall_analysis_cloud' if DATA_SOURCE == 'google_cloud' else 'rainfall_analysis')
CITIES = ['paris','luzern','hamburg','toronto','essen','innsbruck','darmstadt','manchester']
NAMES = {'paris':'Paris','luzern':'Luzern','hamburg':'Hamburg','toronto':'Toronto',
         'essen':'Essen','innsbruck':'Innsbruck','darmstadt':'Darmstadt','manchester':'Manchester'}
BINS = [0, DRY_THRESHOLD_MM, EVENT_PEAK_MIN_MM, 4, 10, float('inf')]
LABELS = ['Dry','Light','Moderate','Heavy','Extreme']
MIN_DRY = 3
MIN_SENSORS = 5
MIN_PANEL_SHARE = .5
MIN_SUPPORTED_HOURS = 10
MIN_SUPPORTED_DATES = 3
PRE_HOURS = 3
MIN_POST_HOURS = 3
MAX_POST_HOURS = 6
RECOVERY_TOLERANCE = 10  # percentage points around expected dry flow
KEYS = ['detid','hour','is_weekend']
# ERA5 rain totals end at their validity timestamp; traffic hours are labelled
# by their start in the public hourly aggregation code. Shift rain back 1 hour.
RAIN_END_TO_START_HOURS = 1

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

In [69]:
# @title Load selected city data from Google Cloud
def read_cloud_cities(stream, chunksize=250_000):
    required = ['city','detid','timestamp','flow','speed','occupancy','precipitation']
    selected, row_counts = [], {}
    reader = pd.read_csv(stream, usecols=required, chunksize=chunksize,
                         dtype={'city':'string','detid':'string'})
    for number, chunk in enumerate(reader, 1):
        chunk['city'] = chunk.city.str.lower().str.strip()
        for city, count in chunk.city.value_counts().items():
            row_counts[city] = row_counts.get(city, 0) + int(count)
        chosen = chunk[chunk.city.isin(CITIES)].copy()
        if not chosen.empty:
            selected.append(chosen)
        if number % 5 == 0:
            print(f'Scanned {sum(row_counts.values()):,} cloud rows...', flush=True)
    if not selected:
        raise ValueError('The source contains none of the selected cities.')
    data = pd.concat(selected, ignore_index=True)
    missing = sorted(set(CITIES) - set(data.city.unique()))
    if missing:
        raise ValueError(f'Selected cities missing from cloud source: {missing}')
    data['timestamp'] = pd.to_datetime(data.timestamp, errors='raise')
    data['flow'] = pd.to_numeric(data.flow, errors='raise')
    data['precipitation'] = pd.to_numeric(data.precipitation, errors='raise')
    if data.duplicated(['city','detid','timestamp']).any():
        raise ValueError('Duplicate city–sensor–hour records; inspect before computing baselines.')
    # The merged export repeats city rainfall for each sensor; conflicting values are an error.
    if data.groupby(['city','timestamp']).precipitation.nunique(dropna=False).gt(1).any():
        raise ValueError('Conflicting precipitation within city-hours.')
    summary = pd.DataFrame(sorted(row_counts.items()), columns=['city','source_rows'])
    return data, summary

if DATA_SOURCE == 'google_cloud':
    with urlopen(DATA_URL, timeout=60) as stream:
        cloud_df, cloud_city_counts = read_cloud_cities(stream)
    print('Loaded from:', DATA_URL)
    print(f'{cloud_city_counts.source_rows.sum():,} source rows; {len(cloud_city_counts)} source cities; {len(cloud_df):,} selected sensor-hours')
    display(cloud_city_counts)
    display(cloud_df.head())
    display(cloud_df.groupby('city').agg(sensor_hours=('timestamp','size'),
        observed_hours=('timestamp','nunique'),
        observed_dates=('timestamp',lambda s:s.dt.date.nunique()),
        start=('timestamp','min'),end=('timestamp','max')))
else:
    if DATA_SOURCE != 'raw_parquet':
        raise ValueError('DATA_SOURCE must be google_cloud or raw_parquet.')
    print('Optional raw-Parquet mode: expecting IUTFD/ under', ROOT)

Scanned 1,250,000 cloud rows...
Scanned 2,500,000 cloud rows...
Scanned 3,750,000 cloud rows...
Scanned 5,000,000 cloud rows...
Scanned 6,250,000 cloud rows...
Scanned 7,500,000 cloud rows...
Scanned 8,750,000 cloud rows...
Scanned 10,000,000 cloud rows...
Loaded from: https://storage.googleapis.com/bucket-quickstart-iutfdataset/all_cities_combined.csv
10,489,938 source rows; 38 source cities; 4,580,103 selected sensor-hours


,city,source_rows
0,augsburg,271144
1,basel,13013
2,bern,127283
3,birmingham,10332
4,bolton,20750
5,bordeaux,70368
6,bremen,178972
7,cagliari,142125
8,constance,20496
9,darmstadt,266951


,city,detid,timestamp,flow,speed,occupancy,precipitation
0,darmstadt,K102D2,2015-09-21,15.0,NaN,0.003000,0.0
1,darmstadt,K13D11,2015-09-21,19.0,NaN,0.004000,0.0
2,darmstadt,K13D12,2015-09-21,8.0,NaN,0.001667,0.0
3,darmstadt,K13D21,2015-09-21,9.0,NaN,0.001333,0.0
4,darmstadt,K13D22,2015-09-21,3.0,NaN,0.000167,0.0


,sensor_hours,observed_hours,observed_dates,start,end
city,,,,,
darmstadt,266951,900,39,2015-09-21 00:00:00,2016-05-06 23:00:00
essen,29663,824,35,2017-03-27 10:00:00,2017-09-30 23:00:00
hamburg,1022272,2508,105,2016-08-27 00:00:00,2016-12-09 11:00:00
innsbruck,11520,720,30,2017-04-01 00:00:00,2017-04-30 23:00:00
luzern,1344504,8759,366,2015-01-01 00:00:00,2016-01-01 00:00:00
manchester,96199,588,28,2017-09-08 08:00:00,2017-11-18 12:00:00
paris,1548314,8036,336,2016-01-01 01:00:00,2016-12-01 00:00:00
toronto,260680,1464,61,2016-09-01 00:00:00,2017-01-31 23:00:00


## 2. Prepare rainfall and traffic measurements

Multiply each hour's precipitation in metres by 1,000 to obtain mm/hour. Rebuild the rainfall classes from those values. Explicitly missing rainfall remains **Unknown**, and negative or nonfinite flow is treated as missing. The flow field is the supplied mean flow.

| Rainfall class | Hourly precipitation (mm/hour) |
| --- | --- |
| Dry | 0 to less than 0.1 |
| Light | 0.1 to less than 0.5 |
| Moderate | 0.5 to less than 4 |
| Heavy | 4 to less than 10 |
| Extreme | 10 or more |
| Unknown | Missing precipitation |

In raw mode, the weather timestamp is shifted back one hour so the end of the rainfall accumulation interval aligns with the traffic hour's start. The merged cloud timestamps are used as supplied.

In [70]:
# @title Optional: load raw Parquet data
def load_raw_city(city):
    path = ROOT / 'IUTFD' / city
    d = pd.read_parquet(path/'sensors/hourly_readings.parquet',
                        columns=['datetime','detid','flow_mean_5min'])
    d = d.rename(columns={'flow_mean_5min':'flow'})
    d['timestamp'] = pd.to_datetime(d.pop('datetime'),format='mixed',dayfirst=True)
    d['detid'] = d.detid.astype(str)
    assert not d.duplicated(['detid','timestamp']).any(), city
    d['flow'] = d.flow.where(d.flow.notna() & ~d.flow.isin([float('inf'),float('-inf')]) & d.flow.ge(0))
    w = pd.concat([pd.read_parquet(p,columns=['local_time','utc_time','grid_id','total_precipitation'])
                   for p in sorted((path/'weather/datetime').glob('*.parquet'))],ignore_index=True)
    assert not w.duplicated(['utc_time','grid_id']).any(), city
    w['rain_timestamp'] = pd.to_datetime(w.local_time,format='mixed') - pd.Timedelta(hours=RAIN_END_TO_START_HOURS)
    assert w.total_precipitation.dropna().ge(0).all(), city
    rain = w.groupby('rain_timestamp').total_precipitation.mean().mul(METRES_TO_MM)
    # Do not fill failed weather matches with 0.
    d['rain_mm'] = d.timestamp.map(rain)
    d['rain_bin'] = pd.cut(d.rain_mm,BINS,labels=LABELS,right=False)
    d['hour'] = d.timestamp.dt.hour
    d['is_weekend'] = d.timestamp.dt.dayofweek.ge(5)
    d['city'] = city
    return d



def load_city(city):
    if DATA_SOURCE == 'raw_parquet':
        return load_raw_city(city)
    d = cloud_df.loc[cloud_df.city.eq(city), ['city','detid','timestamp','flow','precipitation']].copy()
    d['detid'] = d.detid.astype(str)
    d['flow'] = d.flow.where(d.flow.notna() & ~d.flow.isin([float('inf'),float('-inf')]) & d.flow.ge(0))
    d['rain_mm'] = d.pop('precipitation').mul(METRES_TO_MM)
    if (d.rain_mm.lt(0) | d.rain_mm.eq(float('inf'))).any():
        raise ValueError(f'Nonfinite or negative precipitation in {city}.')
    d['rain_bin'] = pd.cut(d.rain_mm, BINS, labels=LABELS, right=False)
    d['hour'] = d.timestamp.dt.hour
    d['is_weekend'] = d.timestamp.dt.dayofweek.ge(5)
    return d

## 3. Build dry baselines and city-hour summaries

For each city and sensor, compute mean dry flow for the same recorded hour of day and day type. The reference needs at least three valid dry observations and a positive mean. For a dry observation, leave that observation out of its own reference.

`Flow deviation (%) = 100 × (observed flow / matched dry mean − 1)`

Each city-hour estimate is the median deviation across supported sensors. It requires at least five supported sensors covering at least half of all sensors recorded that hour. The denominator includes sensors with missing flow, and the set of supported sensors can vary between hours.

In [71]:
# @title Build dry baselines and hourly summaries
def build_baseline(d):
    dry = d[d.rain_bin.eq('Dry') & d.flow.notna()]
    base = dry.groupby(KEYS).flow.agg(dry_sum='sum',dry_n='count').reset_index()
    d = d.merge(base,on=KEYS,how='left',validate='many_to_one')
    # Leave each dry observation out of its own comparison baseline.
    own_dry = d.rain_bin.eq('Dry') & d.flow.notna()
    n = d.dry_n - own_dry.astype(int)
    total = d.dry_sum - d.flow.where(own_dry,0)
    d['dry_baseline'] = total.div(n).where(n.ge(MIN_DRY)&total.gt(0))
    d['pct_flow_change'] = 100*(d.flow/d.dry_baseline-1)
    assert not d.pct_flow_change.dropna().isin([float('inf'),float('-inf')]).any()
    return d


def summarize_hours(d):
    h = d.groupby('timestamp').agg(rain_mm=('rain_mm','first'),
          sensors_observed=('detid','nunique'),supported_sensor_rows=('pct_flow_change','count'),
          median_flow_change_pct=('pct_flow_change','median')).reset_index()
    h['rain_bin'] = pd.cut(h.rain_mm,BINS,labels=LABELS,right=False)
    h['supported_sensor_share'] = h.supported_sensor_rows/h.sensors_observed
    # A changing citywide panel is retained only for descriptive intensity
    # summaries. Event curves below use a strictly fixed panel per event.
    h['median_flow_change_pct'] = h.median_flow_change_pct.where(
        h.supported_sensor_rows.ge(MIN_SENSORS)&h.supported_sensor_share.ge(MIN_PANEL_SHARE))
    h['city'] = d.city.iloc[0]
    h['date'] = h.timestamp.dt.date.astype(str)
    return h

## 4. Classify rain events and recovery

1. Detect continuous spells at or above 0.1 mm/hour with at least one hour at or above 0.5 mm/hour. Light rain can form the beginning or end of a spell.
2. Require three observed dry hours before the spell and at least three after it. Follow-up continues for up to six consecutive observed dry hours.
3. Hold the event window's dry observations out of the baseline. Keep a fixed panel of at least five sensors with valid comparisons throughout the window, covering at least half the sensors recorded in that window.
4. Assess recovery when the median panel deviation reaches −10% or lower during rain. Confirm recovery at the end of the second consecutive post-rain hour within ±10% of baseline. The earliest possible confirmation is two hours after rain ends.

Qualifying events have one of three outcomes: no measured deficit, confirmed recovery, or recovery unconfirmed within follow-up. Unconfirmed events retain a blank recovery time.

In [72]:
# @title Detect rain events and measure recovery
def extract_events(d,h):
    city=d.city.iloc[0]
    weather=h.set_index('timestamp').rain_mm.sort_index()
    records, curves = [], []
    for start,value in weather.items():
        if pd.isna(value) or value < DRY_THRESHOLD_MM: continue
        previous=start-pd.Timedelta(hours=1)
        if previous not in weather.index or not weather.loc[previous] < DRY_THRESHOLD_MM: continue
        end=start
        while end in weather.index and pd.notna(weather.loc[end]) and weather.loc[end] >= DRY_THRESHOLD_MM:
            end+=pd.Timedelta(hours=1)
        wet_times=pd.date_range(start,end-pd.Timedelta(hours=1),freq='h')
        if weather.reindex(wet_times).max() < EVENT_PEAK_MIN_MM: continue
        before=pd.date_range(start-pd.Timedelta(hours=PRE_HOURS),periods=PRE_HOURS,freq='h')
        pre=weather.reindex(before)
        after=pd.date_range(end,periods=MIN_POST_HOURS,freq='h')
        post=weather.reindex(after)
        if not (pre.notna().all() and pre.lt(DRY_THRESHOLD_MM).all() and post.notna().all() and post.lt(DRY_THRESHOLD_MM).all()):continue
        # Extend follow-up only while every subsequent hour is observed and dry.
        post_n=MIN_POST_HOURS
        while post_n<MAX_POST_HOURS:
            t=end+pd.Timedelta(hours=post_n)
            if t not in weather.index or pd.isna(weather.loc[t]) or weather.loc[t] >= DRY_THRESHOLD_MM:break
            post_n+=1
        times=pd.date_range(before[0],end+pd.Timedelta(hours=post_n-1),freq='h')
        window=d[d.timestamp.isin(times)].copy()
        # Withhold all dry observations from this event window from its baseline.
        held=window[window.rain_bin.eq('Dry') & window.flow.notna()].groupby(KEYS).flow.agg(held_sum='sum',held_n='count').reset_index()
        window=window.merge(held,on=KEYS,how='left',validate='many_to_one')
        counts=window.dry_n-window.held_n.fillna(0)
        sums=window.dry_sum-window.held_sum.fillna(0)
        window['event_baseline']=sums.div(counts).where(counts.ge(MIN_DRY)&sums.gt(0))
        window['event_change']=100*(window.flow/window.event_baseline-1)
        panel=window.pivot(index='detid',columns='timestamp',values='event_change').reindex(columns=times).dropna()
        all_sensors=window.detid.nunique()
        if len(panel)<MIN_SENSORS or len(panel)/all_sensors<MIN_PANEL_SHARE:continue
        series=panel.median(axis=0)
        wet_change=series.reindex(wet_times)
        # Report recovery only for episodes with a >=10% flow deficit.
        # Round for threshold comparisons only to absorb floating-point error.
        disruption = bool(wet_change.round(10).le(-RECOVERY_TOLERANCE).any())
        post_times=pd.date_range(end,periods=post_n,freq='h')
        within=series.reindex(post_times).abs().round(10).le(RECOVERY_TOLERANCE).tolist()
        recovered=float('nan')
        if disruption:
            for i in range(post_n-1):
                if within[i] and within[i+1]:
                    # Confirmation requires two completed dry-hour observations.
                    recovered=i+2;break
        event_id=f'{city}_{start:%Y%m%dT%H%M}'
        records.append({'city':city,'event_id':event_id,'start':start,'rain_end':end,
            'duration_hours':len(wet_times),'peak_rain_mm':weather.reindex(wet_times).max(),
            'panel_sensors':len(panel),'panel_share':len(panel)/all_sensors,
            'followup_hours':post_n,'minimum_wet_flow_change_pct':wet_change.min(),
            'flow_deficit_observed':disruption,'recovery_confirmed_hours':recovered,
            'recovery_status':('no measured deficit' if not disruption else
                               'confirmed within follow-up' if pd.notna(recovered) else
                               'not confirmed within follow-up')})
        # Align on end of rain. Negative offsets can include pre-event hours
        # for short episodes, so they are not universally labelled rainy.
        for offset in range(-3,post_n):
            t=end+pd.Timedelta(hours=offset)
            if t in series.index:
                curves.append({'city':city,'event_id':event_id,'hours_from_rain_end':offset,
                               'flow_change_pct':series.loc[t]})
    return records, curves

In [73]:
# @title Process selected cities and build dashboard tables
def main():
    OUT.mkdir(parents=True, exist_ok=True)
    all_hours, all_events, all_curves = [], [], []

    for city in CITIES:
        print(f"Processing city: {city}...", flush=True)
        city_data = load_city(city)
        if city_data.empty:
            print(f"No observations for {city}; skipping.", flush=True)
            continue
        city_data = build_baseline(city_data)
        city_hours = summarize_hours(city_data)
        all_hours.append(city_hours)
        city_events, city_curves = extract_events(city_data, city_hours)
        all_events.extend(city_events)
        all_curves.extend(city_curves)

    hours_columns = ['city','timestamp','rain_mm','rain_bin','median_flow_change_pct','date']
    event_columns = ['city','event_id','start','rain_end','duration_hours','peak_rain_mm',
                     'panel_sensors','panel_share','followup_hours','minimum_wet_flow_change_pct',
                     'flow_deficit_observed','recovery_confirmed_hours','recovery_status']
    curve_columns = ['city','event_id','hours_from_rain_end','flow_change_pct']
    hours = pd.concat(all_hours, ignore_index=True) if all_hours else pd.DataFrame(columns=hours_columns)
    events = pd.DataFrame(all_events, columns=event_columns)
    curves = pd.DataFrame(all_curves, columns=curve_columns)

    coverage, intensity, curve_summary = chart_tables(hours, events, curves)
    recovery = recovery_statistics(events)

    print("Generating and saving interactive Plotly charts...", flush=True)
    coverage_fig, intensity_fig, recovery_fig = make_charts(coverage, intensity, curve_summary, recovery)
    print("Creating dashboard mockup layout sketch...", flush=True)
    dashboard_fig = make_sketch(hours, events, recovery)

    globals().update({
        'hours': hours, 'events': events, 'curves': curves,
        'intensity': intensity, 'recovery': recovery,
        'coverage': coverage, 'curve_summary': curve_summary,
        'coverage_fig': coverage_fig, 'intensity_fig': intensity_fig,
        'recovery_fig': recovery_fig, 'dashboard_fig': dashboard_fig,
    })
    return hours, events, intensity, recovery

## 5. Design the dashboard charts

The dashboard supports three tasks: compare rainfall coverage, examine flow deviations against dry conditions, and follow event-level patterns through the end of rain.

| View | Marks and channels |
| --- | --- |
| Rainfall coverage | Horizontal bars show observed city-hours. Ordered colors distinguish rainfall classes, and a hatch identifies unknown rainfall. |
| Flow by rainfall intensity | Heatmap cells use a symmetric, zero-centered color scale and numeric labels for signed flow deviations. |
| Traffic patterns by hour | Lines show median deviations by clock hour, with bars showing how many city-hours support them. |
| Storm intensity and flow deviation | Points pair each event's peak hourly rainfall with its lowest flow deviation during rain. |
| Flow after rain ends | Aligned city panels show event medians and middle-50% bands on shared axes. |

Following Munzner's what, why, and how framework, the encodings emphasize the distinctions readers need to make: coverage versus composition, direction and magnitude of deviation, and timing relative to rain ending. Labels, patterns, and hover details supplement color.

In [74]:
# @title Build dashboard tables and Plotly figures
import numpy as np

RAIN_COLORS = {'Dry':'#D55E00','Light':'#E69F00','Moderate':'#56B4E9','Heavy':'#0072B2','Extreme':'#332288','Unknown':'#595959'}
FLOW_COLORSCALE = [[0.0, '#B35806'], [0.1, '#D98C3F'], [0.2, '#E5B98C'],
                    [0.3, '#F4DDB8'], [0.4, '#FAF2E5'], [0.5, '#F7F7F7'],
                    [0.6, '#D9EAF7'], [0.7, '#A7CDE3'], [0.8, '#67A9CF'],
                    [0.9, '#3282BD'], [1.0, '#2166AC']]
FLOW_LINE_COLOR = '#005a9c'
FLOW_BAND_COLOR = 'rgba(0,90,156,.18)'
COLORBLIND_SIMULATION_MATRICES = {
    'Protanopia': np.array([[.56667,.43333,0],[.55833,.44167,0],[0,.24167,.75833]]),
    'Deuteranopia': np.array([[.625,.375,0],[.7,.3,0],[0,.3,.7]]),
    'Tritanopia': np.array([[.95,.05,0],[0,.43333,.56667],[0,.475,.525]]),
}

def hex_rgb(hex_color):
    value = hex_color.lstrip('#')
    return np.array([int(value[i:i+2],16) for i in (0,2,4)], dtype=float)

def verify_rain_palette_separation(minimum_distance=35):
    colors = list(RAIN_COLORS.values())
    results = {}
    for name, matrix in COLORBLIND_SIMULATION_MATRICES.items():
        simulated = np.array([hex_rgb(color) @ matrix.T for color in colors])
        distances = [np.linalg.norm(simulated[i] - simulated[j])
                     for i in range(len(colors)) for j in range(i + 1, len(colors))]
        minimum = float(min(distances))
        results[name] = {'minimum_distance': round(minimum, 1),
                         'passed': minimum >= minimum_distance}
    return results

COLORBLIND_PALETTE_RESULTS = verify_rain_palette_separation()
for cvd_type, result in COLORBLIND_PALETTE_RESULTS.items():
    status = 'PASS' if result['passed'] else 'REVIEW'
    print(f"{cvd_type}: {status}; minimum simulated pair distance={result['minimum_distance']:.1f}")
if not all(result['passed'] for result in COLORBLIND_PALETTE_RESULTS.values()):
    raise RuntimeError('Rainfall palette did not pass the color-blind separation check.')

RAIN_TICK_TEXT = ['Dry (<0.1)', 'Light (0.1-<0.5)', 'Moderate (0.5-<4)', 'Heavy (4-<10)', 'Extreme (>=10)']


def summarize_intensity(hours):
    observed = hours[hours.rain_bin.notna()].groupby(['city','rain_bin'], observed=True).agg(
        observed_city_hours=('timestamp','nunique'), observed_dates=('date','nunique')).reset_index()
    eligible = hours[hours.median_flow_change_pct.notna() & hours.rain_bin.notna()]
    supported = eligible.groupby(['city','rain_bin'], observed=True).agg(
        median_change_pct=('median_flow_change_pct','median'), city_hours=('timestamp','nunique'),
        supported_dates=('date','nunique')).reset_index()
    intensity = observed.merge(supported, on=['city','rain_bin'], how='left', validate='one_to_one')
    intensity['city_hours'] = intensity.city_hours.fillna(0).astype(int)
    intensity['supported_dates'] = intensity.supported_dates.fillna(0).astype(int)
    intensity['display_supported'] = intensity.city_hours.ge(MIN_SUPPORTED_HOURS) & intensity.supported_dates.ge(MIN_SUPPORTED_DATES)
    intensity['median_change_pct'] = intensity.median_change_pct.where(intensity.display_supported)
    return intensity


def intensity_color_limit(intensity):
    values = intensity.loc[intensity.display_supported, 'median_change_pct'].dropna()
    maximum = values.abs().max()
    return max(10.0, float(maximum)) if pd.notna(maximum) else 10.0


def coverage_insight(coverage):
    counts = coverage.reindex(columns=LABELS + ['Unknown'], fill_value=0)
    totals = counts.sum(axis=1)
    overall = int(totals.sum())
    if overall == 0:
        return 'No observed city-hours are available under the current filters.'
    city = totals.idxmax()
    unknown_pct = 100 * int(counts['Unknown'].sum()) / overall
    return f'{NAMES[city]} has the most observed hours ({int(totals.loc[city]):,}); {unknown_pct:.1f}% of observed hours have unknown rainfall.'


def intensity_insight(intensity):
    supported = intensity[intensity.rain_bin.ne('Dry') & intensity.display_supported
                          & intensity.median_change_pct.notna()].copy()
    if supported.empty:
        return f'No rainy estimate meets the support rule: at least {MIN_SUPPORTED_HOURS} baseline-supported hours on {MIN_SUPPORTED_DATES} dates.'
    row = supported.loc[supported.median_change_pct.abs().idxmax()]
    return (f'Supported rainy-cell highlight: {NAMES[row.city]} — {row.rain_bin} rain '
            f'(median {row.median_change_pct:+.1f}%; {int(row.city_hours)} hours across '
            f'{int(row.supported_dates)} dates). This is the largest eligible value shown, '
            f'not an estimate for all rain or for unsupported cells.')

def recovery_statistics(events, cities=CITIES):
    rows = []
    for city in cities:
        city_events = events[events.city.eq(city)]
        if 'flow_deficit_observed' in city_events:
            deficits = city_events.flow_deficit_observed.fillna(False)
            if not pd.api.types.is_bool_dtype(deficits.dtype):
                deficits = deficits.astype(str).str.lower().isin(['true','1'])
        else:
            deficits = pd.Series(False, index=city_events.index)
        deficit_count = int(deficits.sum())
        if 'recovery_confirmed_hours' in city_events:
            confirmed = deficits & city_events.recovery_confirmed_hours.notna()
            confirmed_count = int(confirmed.sum())
            recovery_times = city_events.loc[confirmed, 'recovery_confirmed_hours']
        else:
            confirmed_count = 0
            recovery_times = pd.Series(dtype=float)
        rows.append({'city':city, 'well_observed_events':len(city_events),
            'events_with_deficit':deficit_count, 'confirmed_recoveries':confirmed_count,
            'not_confirmed_in_window':deficit_count-confirmed_count,
            'median_recovery_hours':recovery_times.median()})
    return pd.DataFrame(rows).set_index('city')


def recovery_insight(recovery):
    events = int(recovery.well_observed_events.sum())
    deficits = int(recovery.events_with_deficit.sum())
    confirmed = int(recovery.confirmed_recoveries.sum())
    if events == 0:
        message = 'No qualifying rain events are available under the current filters.'
    elif deficits == 0:
        message = 'No selected event showed a measured flow deficit of at least 10%; recovery was not assessed.'
    else:
        event_label = 'event' if deficits == 1 else 'events'
        message = (f'Recovery confirmed within follow-up for {confirmed} of {deficits} '
                   f'{event_label} with a measured flow deficit of at least 10%.')
    return message + ' The shaded band is the middle 50% of observed events, not a confidence interval.'


def chart_tables(hours, events, curves):
    coverage = hours.groupby(['city','rain_bin'], observed=False).size().unstack(fill_value=0).reindex(
        index=CITIES, columns=LABELS, fill_value=0)
    coverage['Unknown'] = hours[hours.rain_bin.isna()].groupby('city').size().reindex(CITIES, fill_value=0)
    intensity = summarize_intensity(hours)
    curve_summary = curves.groupby(['city','hours_from_rain_end']).flow_change_pct.agg(
        median='median', q25=lambda values: values.quantile(.25),
        q75=lambda values: values.quantile(.75), events='count').reset_index()
    return coverage, intensity, curve_summary


def make_coverage_chart(coverage, cities=CITIES, show_insight=True):
    coverage = coverage.reindex(index=cities, columns=LABELS + ['Unknown'], fill_value=0)
    totals = coverage.sum(axis=1)
    percentages = coverage.div(totals.where(totals.ne(0)), axis=0).fillna(0).mul(100)
    labels = LABELS + ['Unknown']
    fig = go.Figure()
    for label in labels:
        fig.add_trace(go.Bar(name=label, y=[NAMES[c] for c in cities], x=coverage[label], orientation='h',
            marker=dict(color=RAIN_COLORS[label], pattern=dict(shape='/' if label == 'Unknown' else '')),
            customdata=pd.concat([coverage[label], percentages[label]], axis=1).values.tolist(),
            hovertemplate='%{y}<br>' + label + ': %{customdata[0]:,.0f} observed city-hours'
                + '<br>%{customdata[1]:.1f}% of this city\'s observed hours'
                + '<br>Unknown weather is not classified as dry.<extra></extra>'))
    for city in cities:
        if totals.loc[city] == 0:
            fig.add_annotation(x=0, y=NAMES[city], text='No observations', showarrow=False,
                               xanchor='left', xshift=5, font=dict(color='#404040', size=11))
    hover_hours = ['%{y}<br>' + label + ': %{customdata[0]:,.0f} observed city-hours'
        + '<br>%{customdata[1]:.1f}% of this city\'s observed hours'
        + '<br>Unknown weather is not classified as dry.<extra></extra>' for label in labels]
    hover_percent = ['%{y}<br>' + label + ': %{customdata[1]:.1f}% of this city\'s observed hours'
        + '<br>Equivalent to %{customdata[0]:,.0f} observed city-hours'
        + '<br>Unknown weather is not classified as dry.<extra></extra>' for label in labels]
    buttons = [
        dict(label='Hours', method='update',
            args=[{'x':[coverage[label].tolist() for label in labels], 'hovertemplate':hover_hours},
                  {'xaxis.title.text':'Observed city-hours', 'xaxis.ticksuffix':'', 'xaxis.autorange':True}]),
        dict(label='Percentage', method='update',
            args=[{'x':[percentages[label].tolist() for label in labels], 'hovertemplate':hover_percent},
                  {'xaxis.title.text':'Share of observed city-hours (%)', 'xaxis.range':[0,100],
                   'xaxis.ticksuffix':'%', 'xaxis.autorange':False}])]
    title = 'Rainfall coverage by city'
    if show_insight:
        title += f'<br><sup>{coverage_insight(coverage)}</sup>'
    fig.update_layout(title=dict(text=title, x=.01, xanchor='left'), template='plotly_white',
        barmode='stack', height=max(520,64*len(cities)), margin=dict(t=150,b=85,l=110,r=35),
        xaxis_title='Observed city-hours', yaxis=dict(autorange='reversed'),
        legend=dict(title='Hourly rainfall', orientation='h', y=1.08),
        updatemenus=[dict(type='buttons',direction='right',x=1,xanchor='right',y=1.24,
                          buttons=buttons,showactive=True)])
    return fig


def make_intensity_chart(intensity, cities=CITIES, color_limit=None, show_insight=True):
    lookup = intensity.set_index(['city','rain_bin'])
    z = pd.DataFrame(float('nan'), index=cities, columns=LABELS)
    text = pd.DataFrame('', index=cities, columns=LABELS)
    info = [[[0,0,0,'No observed hours in this class','Not estimated'] for _ in LABELS] for _ in cities]
    for i, city in enumerate(cities):
        for j, rain_bin in enumerate(LABELS):
            if (city,rain_bin) not in lookup.index:
                text.iat[i,j] = 'NR'
                continue
            item = lookup.loc[(city,rain_bin)]
            if bool(item.display_supported):
                z.iat[i,j] = item.median_change_pct
                status = 'Supported estimate'
                value_label = f'{item.median_change_pct:+.1f}%'
                text.iat[i,j] = value_label
            elif item.city_hours == 0:
                status = 'Observed hours, no baseline support'
                value_label = 'Not estimated'
                text.iat[i,j] = 'NB'
            else:
                status = 'Insufficient hours or date support'
                value_label = 'Not estimated'
                text.iat[i,j] = 'SP'
            info[i][j] = [item.observed_city_hours,item.city_hours,item.supported_dates,status,value_label]
    limit = intensity_color_limit(intensity) if color_limit is None else float(color_limit)
    fig = go.Figure(go.Heatmap(z=[z.loc[c].tolist() for c in cities], x=LABELS,
        y=[NAMES[c] for c in cities], zmin=-limit, zmax=limit, zmid=0, zauto=False,
        colorscale=FLOW_COLORSCALE, customdata=info, hoverongaps=True,
        colorbar=dict(title='Flow (%)',tickvals=[-limit,0,limit],len=.65,thickness=12,
                      tickfont=dict(size=9),ticktext=[f'{-limit:g}%','0%',f'+{limit:g}%']),
        hovertemplate='%{y} | %{x}<br>Status: %{customdata[3]}'
            + '<br>Median flow change: %{customdata[4]}'
            + '<br>Baseline-supported hours: %{customdata[1]:.0f}'
            + '<br>Observed hours in this class: %{customdata[0]:.0f}'
            + '<br>Dates with supported estimates: %{customdata[2]:.0f}<extra></extra>'))
    for i, city in enumerate(cities):
        for j, rain_bin in enumerate(LABELS):
            fig.add_annotation(x=rain_bin,y=NAMES[city],text=text.iat[i,j],showarrow=False,font=dict(size=8,color='#222222'))
    for city, rain_bin in [('innsbruck', 'Dry'), ('luzern', 'Heavy')]:
        if city in z.index and pd.notna(z.at[city, rain_bin]):
            fig.add_annotation(x=rain_bin, y=NAMES[city], text='', showarrow=True,
                               ax=16, ay=-16, standoff=8, arrowhead=2,
                               arrowwidth=1.8, arrowcolor='#453221')
    title = 'Flow change by rain intensity'
    if show_insight:
        title += f'<br><sup>{intensity_insight(intensity)}</sup>'
    fig.update_layout(title=dict(text=title,x=.01,xanchor='left'),template='plotly_white',
        plot_bgcolor='#d9d9d9',height=max(520,72*len(cities)),margin=dict(t=135,b=130,l=85,r=55),
        xaxis=dict(title='Rain class',tickmode='array',tickvals=LABELS,
                   ticktext=['Dry','Light','Mod.','Heavy','Extreme'],tickangle=0,tickfont=dict(size=8)),
        yaxis=dict(title='',autorange='reversed'),
        annotations=list(fig.layout.annotations)+[dict(x=0,y=-.34,xref='paper',yref='paper',
            text='NR: no observations; NB: no baseline;<br>SP: sparse (<10h or <3 dates). Grey is not zero.',
            showarrow=False,xanchor='left',font=dict(size=9,color='#404040'))])
    return fig


def make_recovery_chart(curve_summary, recovery, cities=CITIES, show_insight=True,
                        selected_event=None, event_curves=None):
    cols=min(2,len(cities))
    rows=max(1,(len(cities)-1)//cols+1)
    titles=[f'{NAMES[c]} · {int(recovery.loc[c,"well_observed_events"])}' for c in cities]
    fig=make_subplots(rows=rows,cols=cols,shared_xaxes=True,shared_yaxes=True,
        subplot_titles=titles,horizontal_spacing=.05,vertical_spacing=.16)
    for i,city in enumerate(cities):
        r,c=i//cols+1,i%cols+1
        series=curve_summary[curve_summary.city.eq(city)].sort_values('hours_from_rain_end')
        if not series.empty:
            fig.add_trace(go.Scatter(x=series.hours_from_rain_end,y=series.q25,mode='lines',
                line=dict(width=0,color=FLOW_BAND_COLOR),showlegend=False,hoverinfo='skip'),row=r,col=c)
            fig.add_trace(go.Scatter(x=series.hours_from_rain_end,y=series.q75,mode='lines',
                line=dict(width=0,color=FLOW_BAND_COLOR),fill='tonexty',fillcolor=FLOW_BAND_COLOR,
                name='Middle 50% of events',legendgroup='event-spread',showlegend=i==0,hoverinfo='skip'),row=r,col=c)
            custom=series[['q25','q75','events']].to_numpy()
            fig.add_trace(go.Scatter(x=series.hours_from_rain_end,y=series['median'],mode='lines+markers',
                line=dict(color=FLOW_LINE_COLOR,width=2),marker=dict(size=6),name='Event median',
                legendgroup='event-median',showlegend=i==0,customdata=custom,
                hovertemplate=NAMES[city]+'<br>Hours from rain end: %{x}'
                    +'<br>Median flow change: %{y:+.1f}%'
                    +'<br>Middle 50%: %{customdata[0]:+.1f}% to %{customdata[1]:+.1f}%'
                    +'<br>Contributing events: %{customdata[2]:.0f}<extra></extra>'),row=r,col=c)
        if selected_event and selected_event['city'] == city and event_curves is not None:
            selected_series = event_curves[event_curves.event_id.eq(selected_event['event_id'])].sort_values(
                'hours_from_rain_end')
            if not selected_series.empty:
                event_date = pd.to_datetime(selected_event['start']).strftime('%Y-%m-%d')
                fig.add_trace(go.Scatter(x=selected_series.hours_from_rain_end,
                    y=selected_series.flow_change_pct,mode='lines+markers',
                    line=dict(color='#D1495B',width=3),marker=dict(size=8,symbol='diamond'),
                    name=f'Selected event · {event_date}',legendgroup='selected-event',showlegend=True,
                    hovertemplate=f'Selected event · {event_date}<br>Hours from rain end: %{{x}}'
                        +'<br>Flow change: %{y:+.1f}%<extra></extra>'),row=r,col=c)
        fig.add_hrect(y0=-RECOVERY_TOLERANCE,y1=RECOVERY_TOLERANCE,
            fillcolor='#777777',opacity=.10,line_width=0,row=r,col=c)
        fig.add_hline(y=0,line_dash='dash',line_color='#555555',row=r,col=c)
        fig.add_vline(x=0,line_dash='dot',line_color='#555555',row=r,col=c)
    curve_values=[]
    if not curve_summary.empty:
        curve_values.extend(curve_summary[['q25','q75']].to_numpy().ravel())
    if selected_event and event_curves is not None:
        selected_values=event_curves.loc[
            event_curves.event_id.eq(selected_event['event_id']),'flow_change_pct'
        ].to_numpy()
        curve_values.extend(selected_values)
    curve_values=[value for value in curve_values if pd.notna(value)]
    y_min=min([-RECOVERY_TOLERANCE,*curve_values])
    y_max=max([RECOVERY_TOLERANCE,*curve_values])
    y_padding=max(2,(y_max-y_min)*.05)
    fig.update_yaxes(range=[y_min-y_padding,y_max+y_padding])
    title='Recovery after rain ends'
    if show_insight:
        title+=f'<br><sup>{recovery_insight(recovery)}</sup>'
    fig.update_xaxes(range=[-3.2, 6.5],dtick=1)
    fig.update_layout(title=dict(text=title,x=.01,xanchor='left'),template='plotly_white',
        height=max(500,350*rows),margin=dict(l=70,b=180,t=135,r=20),legend=dict(orientation='h',y=-.12),
        annotations=list(fig.layout.annotations)+[
            dict(x=.5,y=-.09,xref='paper',yref='paper',text='Hours from rain end (0 = first dry hour)',showarrow=False),
            dict(x=-.055,y=.5,xref='paper',yref='paper',text='Flow change (%)',textangle=-90,showarrow=False),
            dict(x=.01,y=-.34,xref='paper',yref='paper',
                text=f'Grey band: ±{RECOVERY_TOLERANCE}% of baseline; curve band: middle 50% of events, not a confidence interval.',
                showarrow=False,xanchor='left',font=dict(size=11,color='#404040'))])
    return fig


def make_charts(coverage, intensity, curve_summary, recovery):
    figures=(make_coverage_chart(coverage),make_intensity_chart(intensity),
             make_recovery_chart(curve_summary,recovery))
    for filename,figure in zip(['chart_1_rainfall_coverage.html','chart_2_flow_by_intensity.html','chart_3_event_recovery.html'],figures):
        figure.write_html(OUT/filename,include_plotlyjs=True,full_html=True)
    return figures

In [75]:
# @title Build the exploratory and dashboard Plotly figures
ADDITIONAL_CITY_COLORS = dict(zip(CITIES, [
    '#2b6da8', '#b95b16', '#26816e', '#8c52a4',
    '#a63d59', '#625a2a', '#576e96', '#555555',
]))
ADDITIONAL_CITY_MARKERS = dict(zip(CITIES, [
    'circle', 'square', 'diamond', 'cross', 'x', 'triangle-up', 'triangle-down', 'pentagon',
]))
FLOW_DEVIATION_AXIS = 'Flow deviation from dry baseline (%)'
EVENT_OUTCOME_LABELS = {
    'no measured deficit': 'No measured deficit',
    'confirmed within follow-up': 'Confirmed recovery',
    'not confirmed within follow-up': 'Recovery not confirmed',
}


def style_additional_plot(fig, title, height=460):
    fig.update_layout(
        template='plotly_white', height=height,
        title={'text': title,
               'x': 0.02, 'y': 0.97, 'yanchor': 'top'},
        margin={'l': 68, 'r': 35, 't': 100, 'b': 65},
        font={'size': 13, 'color': '#253447'},
        legend={'orientation': 'h', 'y': 1.02, 'yanchor': 'bottom'},
    )
    return fig


def select_additional_plot_group(fig, groups, active, group_layouts=None):
    # Each group contains the traces for one dropdown choice.
    if not groups:
        fig.add_annotation(text='No qualifying observations', showarrow=False,
                           x=0.5, y=0.5, xref='paper', yref='paper')
        return
    buttons = []
    for label, indices in groups.items():
        visible = [index in indices for index in range(len(fig.data))]
        layout = group_layouts[label] if group_layouts else {}
        buttons.append(dict(label=label, method='update', args=[{'visible': visible}, layout]))
    for index, trace in enumerate(fig.data):
        trace.visible = index in groups[active]
    if group_layouts:
        fig.update_layout(group_layouts[active])
    fig.update_layout(
        updatemenus=[dict(buttons=buttons, active=list(groups).index(active),
                          x=1, xanchor='right', y=1.19, yanchor='bottom')],
        margin_t=132,
    )

def add_flow_range_control(fig):
    # A few large ratios obscure the main distribution on an automatic scale.
    fig.update_yaxes(range=[-105, 100])
    fig.update_layout(height=fig.layout.height + 52, margin_b=117)
    fig.layout.updatemenus += (dict(
        type='buttons', direction='right', x=1, xanchor='right', y=-0.18, yanchor='top',
        buttons=[
            dict(label='−105% to +100%', method='relayout',
                 args=[{'yaxis.autorange': False, 'yaxis.range': [-105, 100]}]),
            dict(label='Full range', method='relayout', args=[{'yaxis.autorange': True}]),
        ],
    ),)

DASH_COMPARISON_CHARTS = [
    ('hourly', 'Traffic patterns by hour of day',
     'Compare dry and rainy medians and supported city-hour counts. The profile always includes all rainfall intensities, even when another chart is filtered.'),
    ('storm-disruption', 'Storm intensity vs. traffic disruption',
     'Each dot is one qualifying rain event. Compare peak hourly rainfall with the lowest flow deviation from the matched dry baseline during rain. Labels mark the deepest flow dip and highest peak rain; click an event to highlight its recovery trajectory.'),
]


def prepare_supported_hours(hours):
    supported = hours.dropna(subset=['rain_mm', 'rain_bin', 'median_flow_change_pct']).copy()
    supported['hour_of_day'] = pd.to_datetime(supported.timestamp).dt.hour
    supported['condition'] = supported.rain_bin.eq('Dry').map({True: 'Dry', False: 'Rainy'})
    return supported


def add_empty_plot_message(fig, message):
    fig.add_annotation(text=message, showarrow=False, x=0.5, y=0.5, xref='paper', yref='paper')


def make_hourly_traffic_chart(supported_hours, city_menu=True):
    fig_hourly = make_subplots(rows=2, cols=1, shared_xaxes=True,
                               row_heights=[0.72, 0.28], vertical_spacing=0.10)
    hourly_groups = {}
    hour_of_day = list(range(24))
    hour_styles = {'Dry': ('#64748b', 'dash', 'circle-open'), 'Rainy': ('#1768ac', 'solid', 'diamond')}
    hourly_samples = [(None, supported_hours)]
    if city_menu:
        hourly_samples.extend(supported_hours.groupby('city', sort=False))
    for city, city_hours in hourly_samples:
        if city_hours.empty:
            continue
        label = NAMES[city] if city else 'All cities'
        first = len(fig_hourly.data)
        for condition, (color, dash, marker) in hour_styles.items():
            summary = (
                city_hours.loc[city_hours.condition.eq(condition)]
                .groupby('hour_of_day').median_flow_change_pct.agg(['median', 'size'])
                .reindex(hour_of_day)
            )
            sample_counts = summary['size'].fillna(0).astype(int)
            fig_hourly.add_trace(go.Scatter(
                x=hour_of_day, y=summary['median'], customdata=sample_counts,
                name=condition, legendgroup=condition, mode='lines+markers', connectgaps=False,
                line=dict(color=color, dash=dash), marker=dict(symbol=marker, size=7),
                hovertemplate='%{x:02d}:00<br>Median %{y:+.2f}%<br>City-hours %{customdata:,}',
            ), row=1, col=1)
            fig_hourly.add_trace(go.Bar(
                x=hour_of_day, y=sample_counts, name=condition, legendgroup=condition, marker_color=color,
                showlegend=False, hovertemplate='%{x:02d}:00<br>City-hours %{y:,}',
            ), row=2, col=1)
        hourly_groups[label] = list(range(first, len(fig_hourly.data)))

    style_additional_plot(fig_hourly, 'Median traffic-flow deviation by hour of day', 560)
    if city_menu and len(hourly_groups) > 1:
        select_additional_plot_group(fig_hourly, hourly_groups, 'All cities')
    profile_stats = {}
    for condition in ('Dry', 'Rainy'):
        profile_stats[condition] = supported_hours.loc[supported_hours.condition.eq(condition)].groupby(
            'hour_of_day').agg(median=('median_flow_change_pct', 'median'),
                city_hours=('timestamp', 'size'), dates=('date', 'nunique')).add_suffix(f'_{condition.lower()}')
    profile = profile_stats['Dry'].join(profile_stats['Rainy'], how='inner')
    profile = profile.loc[
        profile.city_hours_dry.ge(MIN_SUPPORTED_HOURS)
        & profile.dates_dry.ge(MIN_SUPPORTED_DATES)
        & profile.city_hours_rainy.ge(MIN_SUPPORTED_HOURS)
        & profile.dates_rainy.ge(MIN_SUPPORTED_DATES)
    ]
    if profile.empty:
        profile_insight = (
            'No supported dry–rainy gap: each condition needs at least '
            f'{MIN_SUPPORTED_HOURS} city-hours on {MIN_SUPPORTED_DATES} dates.'
        )
    else:
        profile['gap'] = profile.median_rainy - profile.median_dry
        hour = profile.gap.abs().idxmax()
        row = profile.loc[hour]
        profile_insight = (
            f'Largest supported gap: {hour:02d}:00 · rainy {row.median_rainy:+.1f}% vs '
            f'dry {row.median_dry:+.1f}% (Δ {row.gap:+.1f} pp; '
            f'n={int(row.city_hours_dry)}/{int(row.city_hours_rainy)} city-hours).'
        )
    fig_hourly.add_annotation(x=.02, y=1.11, xref='paper', yref='paper',
        text=profile_insight, showarrow=False, xanchor='left', yanchor='bottom',
        font=dict(size=10, color='#404040'))
    fig_hourly.update_layout(margin_t=max(fig_hourly.layout.margin.t or 0, 145))
    fig_hourly.update_layout(barmode='group')
    fig_hourly.update_xaxes(tickvals=hour_of_day[::3] + [23],
                            ticktext=[f'{hour:02d}:00' for hour in hour_of_day[::3] + [23]])
    fig_hourly.update_xaxes(title_text='Recorded hour of day', row=2, col=1)
    fig_hourly.update_yaxes(title_text='Median deviation (%)', zerolinecolor='#64748b', row=1, col=1)
    fig_hourly.update_yaxes(title_text='City-hours', rangemode='tozero', row=2, col=1)
    if not hourly_groups:
        add_empty_plot_message(fig_hourly, 'No supported city-hours for this selection')
    return fig_hourly


def make_rain_flow_scatter(supported_hours, cities=CITIES, city_menu=True):
    fig_rain_scatter = go.Figure()
    rain_scatter_groups = {}
    for city in cities:
        rainy_hours = supported_hours.loc[
            supported_hours.city.eq(city) & supported_hours.condition.eq('Rainy')
        ]
        if rainy_hours.empty:
            continue
        label = NAMES[city]
        rain_scatter_groups[label] = [len(fig_rain_scatter.data)]
        fig_rain_scatter.add_trace(go.Scatter(
            x=rainy_hours.rain_mm.round(5),
            y=rainy_hours.median_flow_change_pct.round(3),
            text=rainy_hours.timestamp.dt.strftime('%Y-%m-%d %H:%M'),
            customdata=rainy_hours.supported_sensor_rows,
            name=label, mode='markers',
            marker=dict(size=6, opacity=0.42, color=ADDITIONAL_CITY_COLORS[city],
                        symbol=ADDITIONAL_CITY_MARKERS[city]),
            hovertemplate='%{text}<br>Rain %{x:.3f} mm/hour<br>Flow deviation %{y:+.2f}%'
                          '<br>Supported sensors %{customdata:,}',
        ))

    style_additional_plot(fig_rain_scatter, 'Rainfall intensity and traffic-flow deviation')
    if city_menu and len(rain_scatter_groups) > 1:
        rain_scatter_groups = {
            'All cities': list(range(len(fig_rain_scatter.data))), **rain_scatter_groups
        }
        select_additional_plot_group(fig_rain_scatter, rain_scatter_groups, 'All cities')
    fig_rain_scatter.update_xaxes(title_text='Hourly rainfall (mm/hour)')
    fig_rain_scatter.update_yaxes(title_text=FLOW_DEVIATION_AXIS, zerolinecolor='#64748b')
    add_flow_range_control(fig_rain_scatter)
    if not fig_rain_scatter.data:
        add_empty_plot_message(fig_rain_scatter, 'No supported rainy hours for this selection')
    return fig_rain_scatter


def make_rainfall_calendar(hours):
    calendar_days = []
    for city, city_hours in hours.groupby('city', sort=False):
        daily = city_hours.groupby(pd.to_datetime(city_hours.timestamp).dt.normalize()).rain_mm.agg(
            observed_hours='count', observed_mm=lambda values: values.sum(min_count=1),
        )
        daily = daily.reindex(pd.date_range(daily.index.min(), daily.index.max(), name='day'))
        daily['observed_hours'] = daily.observed_hours.fillna(0).astype(int)
        daily['city'] = city
        calendar_days.append(daily.reset_index())

    fig_calendar = go.Figure()
    calendar_groups = {}
    calendar_axes = {}
    weekdays = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
    if calendar_days:
        calendar_days = pd.concat(calendar_days, ignore_index=True)
        rainfall_max = calendar_days.observed_mm.max()
        rainfall_max = max(1, rainfall_max) if pd.notna(rainfall_max) else 1
        for (city, year), days in calendar_days.groupby(
            ['city', calendar_days.day.dt.year], sort=False
        ):
            first = len(fig_calendar.data)
            label = f'{NAMES[city]} {year}'
            january = pd.Timestamp(year=int(year), month=1, day=1)
            month_starts = pd.date_range(january, periods=12, freq='MS')
            calendar_axes[label] = {'xaxis.tickvals': (
                (month_starts.dayofyear - 1 + january.dayofweek) // 7
            ).tolist()}
            days = days.assign(
                week=((days.day - january).dt.days + january.dayofweek) // 7,
                weekday=days.day.dt.dayofweek,
            )
            grid_dates = days.pivot(index='weekday', columns='week', values='day').reindex(
                index=range(7), columns=range(54)
            )
            grid_rain = days.pivot(index='weekday', columns='week', values='observed_mm').reindex_like(grid_dates)
            grid_count = days.pivot(index='weekday', columns='week', values='observed_hours').reindex_like(grid_dates)
            date_labels = grid_dates.to_numpy(dtype='datetime64[D]').astype(str)
            heatmap_grid = dict(x=list(range(54)), y=weekdays, text=date_labels,
                                xgap=2, ygap=2, hoverongaps=False, name=label)
            fig_calendar.add_trace(go.Heatmap(
                **heatmap_grid, z=grid_rain.round(3), customdata=grid_count,
                coloraxis='coloraxis',
                hovertemplate='%{text}<br>Observed rain %{z:.2f} mm<br>Hours observed %{customdata}/24<extra>%{fullData.name}</extra>',
            ))
            fig_calendar.add_trace(go.Heatmap(
                **heatmap_grid, z=grid_count.where(grid_count.eq(0)), zmin=0, zmax=1,
                colorscale=[[0, '#aab2bd'], [1, '#aab2bd']], showscale=False,
                hovertemplate='%{text}<br>No observed rainfall amounts (0/24 hours)<extra>%{fullData.name}</extra>',
            ))
            partial = days.loc[days.observed_hours.between(1, 23)]
            fig_calendar.add_trace(go.Scatter(
                x=partial.week, y=partial.weekday.map(dict(enumerate(weekdays))),
                mode='markers', marker=dict(symbol='x', size=7, color='#1f2937',
                                            line=dict(color='white', width=1)),
                hoverinfo='skip', showlegend=False,
            ))
            calendar_groups[label] = list(range(first, len(fig_calendar.data)))

    style_additional_plot(fig_calendar, 'Observed daily rainfall and coverage', 385)
    month_labels = pd.date_range('2024-01-01', periods=12, freq='MS').strftime('%b').tolist()
    fig_calendar.update_xaxes(
        tickmode='array', ticktext=month_labels, range=[-0.5, 53.5], showgrid=False, zeroline=False,
    )
    select_additional_plot_group(fig_calendar, calendar_groups,
                               next(iter(calendar_groups), None), calendar_axes)
    fig_calendar.update_yaxes(
        categoryorder='array', categoryarray=weekdays, autorange='reversed', showgrid=False,
    )
    if calendar_groups:
        fig_calendar.update_layout(coloraxis=dict(
            cmin=0, cmax=rainfall_max, colorscale=[[0, '#e5eff8'], [1, '#1768ac']],
            colorbar=dict(title='Rain (mm)'),
        ))
    return fig_calendar


def make_event_outcomes_chart(events, cities=CITIES):
    event_outcomes = (
        events.groupby(['city', 'recovery_status']).size().unstack(fill_value=0)
        .reindex(index=cities, columns=list(EVENT_OUTCOME_LABELS), fill_value=0)
    )
    event_totals = event_outcomes.sum(axis=1)
    fig_event_outcomes = go.Figure()
    for (status, label), color, pattern in zip(
        EVENT_OUTCOME_LABELS.items(), ['#a1aebd', '#26736d', '#a85b24'], ['', '', '/']
    ):
        fig_event_outcomes.add_trace(go.Bar(
            x=[NAMES[city] for city in cities], y=event_outcomes[status], customdata=event_totals,
            name=label, marker=dict(color=color, pattern=dict(shape=pattern)),
            hovertemplate='%{x}<br>Outcome count %{y} out of %{customdata}<extra>%{fullData.name}</extra>',
        ))
    style_additional_plot(fig_event_outcomes, 'Recovery outcomes for qualifying rain events')
    fig_event_outcomes.update_layout(barmode='stack')
    fig_event_outcomes.update_yaxes(title_text='Qualifying rain events', rangemode='tozero')
    if events.empty:
        add_empty_plot_message(fig_event_outcomes, 'No qualifying rain events for this selection')
    return fig_event_outcomes


def make_flow_distribution_chart(supported_hours):
    fig_flow_boxes = go.Figure()
    box_labels = []
    for rain_class in LABELS:
        flow_group = supported_hours.loc[supported_hours.rain_bin.eq(rain_class)]
        count = len(flow_group)
        box_labels.append(f'{rain_class}<br>n={count:,}')
        if not count:
            continue
        fig_flow_boxes.add_trace(go.Box(
            x0=rain_class, y=flow_group.median_flow_change_pct.round(4),
            name=rain_class, boxpoints='outliers', quartilemethod='linear',
            fillcolor=RAIN_COLORS[rain_class], line_color='#364c68',
            marker=dict(color='#364c68', opacity=0.45, size=4), showlegend=False,
        ))

    style_additional_plot(fig_flow_boxes, 'Traffic-flow deviation by rainfall class', 500)
    fig_flow_boxes.update_xaxes(
        title_text='Rainfall class', type='category', categoryorder='array', categoryarray=LABELS,
        tickvals=LABELS, ticktext=box_labels, range=[-0.5, len(LABELS) - 0.5],
    )
    fig_flow_boxes.update_yaxes(title_text=FLOW_DEVIATION_AXIS, zerolinecolor='#64748b')
    add_flow_range_control(fig_flow_boxes)
    if supported_hours.empty:
        add_empty_plot_message(fig_flow_boxes, 'No supported city-hours for this selection')
    return fig_flow_boxes


def make_event_duration_chart(events, cities=CITIES):
    duration_events = events.loc[
        events.recovery_status.isin(['confirmed within follow-up', 'not confirmed within follow-up'])
    ].copy()
    duration_events['confirmed'] = duration_events.recovery_confirmed_hours.notna()
    duration_events['plotted_hour'] = duration_events.recovery_confirmed_hours.fillna(
        duration_events.followup_hours
    )
    overlaps = duration_events.groupby(['duration_hours', 'plotted_hour'])
    group_sizes = overlaps.event_id.transform('size')
    offsets = (overlaps.cumcount() / (group_sizes - 1).clip(lower=1) - 0.5) * 0.38
    duration_events['plotted_duration'] = duration_events.duration_hours + offsets.where(group_sizes.gt(1), 0)

    fig_event_duration = go.Figure()
    for city in cities:
        city_events = duration_events.loc[duration_events.city.eq(city)]
        if city_events.empty:
            continue
        hover = [
            f'{NAMES[city]} · {event.start:%Y-%m-%d %H:%M}'
            f'<br>Rain duration {event.duration_hours:g} h'
            f'<br>{EVENT_OUTCOME_LABELS[event.recovery_status]}'
            f'<br>{"Confirmed at" if event.confirmed else "Observed through"} {event.plotted_hour:g} h'
            f'<br>Total observed follow-up {event.followup_hours:g} h'
            for event in city_events.itertuples(index=False)
        ]
        fig_event_duration.add_trace(go.Scatter(
            x=city_events.plotted_duration, y=city_events.plotted_hour,
            name=NAMES[city], mode='markers', text=hover, hovertemplate='%{text}<extra></extra>',
            marker=dict(size=9, color=ADDITIONAL_CITY_COLORS[city],
                        symbol=city_events.confirmed.map({True: 'circle', False: 'triangle-up-open'}),
                        line=dict(width=1.5)),
        ))

    style_additional_plot(fig_event_duration, 'Rain duration and recovery after a measured deficit')
    if duration_events.empty:
        add_empty_plot_message(fig_event_duration, 'No qualifying events with a measured deficit')
    fig_event_duration.update_xaxes(title_text='Rain event duration (hours)', tickformat='d')
    fig_event_duration.update_yaxes(title_text='Hours after rain ended', dtick=1, tickformat='d')
    return fig_event_duration


def make_storm_disruption_chart(events, cities=CITIES):
    fig = go.Figure()
    plotted_events = events.loc[events.city.isin(cities)].dropna(
        subset=['peak_rain_mm', 'minimum_wet_flow_change_pct']
    )
    for city in cities:
        city_events = plotted_events.loc[plotted_events.city.eq(city)].dropna(
            subset=['peak_rain_mm', 'minimum_wet_flow_change_pct']
        )
        if city_events.empty:
            continue
        hover = [
            f'{NAMES[city]} · {event.start:%Y-%m-%d %H:%M}'
            f'<br>Peak rain {event.peak_rain_mm:.3f} mm/hour'
            f'<br>Lowest flow deviation {event.minimum_wet_flow_change_pct:+.2f}%'
            f'<br>Rain duration {event.duration_hours:g} h · {event.panel_sensors:g} sensors'
            f'<br>{EVENT_OUTCOME_LABELS[event.recovery_status]}'
            for event in city_events.itertuples(index=False)
        ]
        fig.add_trace(go.Scatter(
            x=city_events.peak_rain_mm, y=city_events.minimum_wet_flow_change_pct,
            name=NAMES[city], mode='markers', text=hover, customdata=city_events.event_id.astype(str),
            hovertemplate='%{text}<extra></extra>',
            marker=dict(size=9, opacity=0.75, color=ADDITIONAL_CITY_COLORS[city],
                        symbol=ADDITIONAL_CITY_MARKERS[city],
                        line=dict(color='white', width=0.8)),
        ))

    style_additional_plot(fig, 'Peak rainfall and lowest flow deviation during rain', 500)
    if fig.data:
        fig.add_hline(y=-RECOVERY_TOLERANCE, line_color='#64748b', line_dash='dash')
        deepest_dip = plotted_events.loc[plotted_events.minimum_wet_flow_change_pct.idxmin()]
        highest_rain = plotted_events.loc[plotted_events.peak_rain_mm.idxmax()]
        same_event = (deepest_dip.city, deepest_dip.event_id) == (highest_rain.city, highest_rain.event_id)
        if same_event:
            labels = [(deepest_dip,
                f'Highest peak rain and deepest flow dip · {NAMES[deepest_dip.city]} '
                f'({deepest_dip.peak_rain_mm:.2f} mm/h, {deepest_dip.minimum_wet_flow_change_pct:+.1f}%)', 55, -45)]
        else:
            labels = [
                (deepest_dip, f'Deepest flow dip · {NAMES[deepest_dip.city]} '
                    f'({deepest_dip.minimum_wet_flow_change_pct:+.1f}%)', 60, -45),
                (highest_rain, f'Highest peak rain · {NAMES[highest_rain.city]} '
                    f'({highest_rain.peak_rain_mm:.2f} mm/h)', -70, -45),
            ]
        for event, label, ax, ay in labels:
            if event.event_id == 'paris_20160102T0000' and event.event_id == deepest_dip.event_id:
                label += '<br>Overnight after New Year’s Day'
                label += '<br>Possible holiday travel patterns'
            fig.add_annotation(x=event.peak_rain_mm, y=event.minimum_wet_flow_change_pct,
                text=label, align='left', showarrow=True, arrowhead=2, ax=ax, ay=ay,
                bgcolor='rgba(255,255,255,.9)', bordercolor='#64748b', borderwidth=1,
                font=dict(size=10, color='#253447'))
    else:
        add_empty_plot_message(fig, 'No qualifying rain events for this selection')
    fig.update_layout(title_text=(
        'Peak rainfall and lowest flow deviation during rain'
        '<br><sup>Click an event to highlight its recovery trajectory.</sup>'
    ), margin_t=118)
    fig.update_xaxes(title_text='Peak hourly rainfall per event (mm/hour)', rangemode='tozero')
    fig.update_yaxes(title_text='Lowest flow deviation during rain (%)',
                     zeroline=True, zerolinecolor='#64748b')
    return fig


### Draft the dashboard layout

The sketch previews the controls, summary cards, and arrangement of the five charts. Section 9 displays the sketch, and Section 10 opens the interactive app.

In [76]:
# @title Create dashboard layout sketch
def make_sketch(hours, events, recovery):
    # Native Plotly layout drawing with a vibrant, modern color scheme
    fig = go.Figure()

    # Color Palette definitions
    BG_COLOR = '#F4F6F9'
    HEADER_TEXT = '#1A365D'
    ACCENT_BLUE = '#2B6CB0'
    ACCENT_TEAL = '#319795'
    CARD_BG = '#FFFFFF'
    TEXT_MUTED = '#4A5568'
    TEXT_DARK = '#2D3748'

    def text(x, y, content, size=12, color=TEXT_DARK, bold=False):
        formatted_content = f'<b>{content}</b>' if bold else content
        fig.add_annotation(x=x, y=y, text=formatted_content.replace('\n','<br>'),
            showarrow=False, xanchor='left', yanchor='top', align='left',
            font=dict(size=size, color=color))

    def box(x, y, width, height, title, subtitle=None, fillcolor=CARD_BG, linecolor='#CBD5E0', title_color=HEADER_TEXT):
        fig.add_shape(type='rect', x0=x, y0=y, x1=x+width, y1=y+height,
            fillcolor=fillcolor, line=dict(color=linecolor, width=1.5))
        text(x+2, y+height-2, title, 13, color=title_color, bold=True)
        if subtitle:
            text(x+2, y+height-(5 if height <= 9 else 7), subtitle, 11, color=TEXT_MUTED)

    # Title & Header
    text(4, 116, 'Rainfall & Traffic Flow Across Eight Cities', 24, HEADER_TEXT, bold=True)
    text(4, 111, 'DASHBOARD SKETCH • Five views of rainfall, traffic flow, and event recovery', 12, ACCENT_BLUE)

    # Filters Section (Colored header band)
    box(4, 98, 132, 9, 'Filters & Controls',
        'City [All Eight ▾]       Day Type [All ▾]       Rain Intensity [All ▾]',
        fillcolor='#EBF8FF', linecolor='#90CDF4', title_color=ACCENT_BLUE)

    # Key KPI Cards (Vibrant backgrounds)
    kpis = [
        ('Selected Cities', str(len(CITIES)), '#EBF8FF', '#3182CE'),
        ('Observed City-Hours', f'{len(hours):,}', '#E6FFFA', '#234E52'),
        ('Well-Observed Events', str(len(events)), '#EDF2F7', '#2D3748'),
        ('Deficit / Recovered Events', f'{int(recovery.events_with_deficit.sum())} / {int(recovery.confirmed_recoveries.sum())}', '#FEFCBF', '#744210')
    ]
    for x, (title, value, bg, border) in zip([4, 38, 72, 106], kpis):
        box(x, 83, 30, 12, title, fillcolor=bg, linecolor=border, title_color=border)
        text(x+2, 87.5, value, 22, color=border, bold=True)

    box(4, 60, 45, 20, '1. Rainfall coverage',
        'Horizontal stacks show city-hours by rainfall class.\nUse the Hours and Percentage controls.')
    box(51, 60, 85, 20, '2. Traffic flow by rainfall intensity',
        'The city-by-intensity heatmap shows median flow deviation.\nGray marks missing or insufficient support.')

    box(4, 38, 65, 19, f'3. {DASH_COMPARISON_CHARTS[0][1]}',
        'Lines and sample counts compare dry and rainy hours.')
    box(71, 38, 65, 19, f'4. {DASH_COMPARISON_CHARTS[1][1]}',
        'Each dot compares a storm’s peak rain with its lowest flow deviation.')

    box(4, 13, 132, 22, '5. Traffic flow after rain ends',
        'City panels show median flow and the middle 50% of events, aligned to rain end.')
    for i, city in enumerate(CITIES):
        row, column = divmod(i, 4)
        x = 7 + column * 32
        y = 15 + (1 - row) * 5.5
        fig.add_shape(type='rect', x0=x, y0=y, x1=x+29, y1=y+4.8,
                      fillcolor='#F7FAFC', line=dict(color='#E2E8F0', width=1.5))
        text(x+1.5, y+3.5, NAMES[city], 10, HEADER_TEXT, bold=True)

    box(4, 2, 132, 8, 'Study limitations',
        'Cloud weather matches may be incomplete. These charts show associations between rainfall and traffic flow.',
        fillcolor='#FFF5F5', linecolor='#FEB2B2', title_color='#9B2C2C')

    fig.update_layout(template='plotly_white', width=1200, height=1160,
        margin=dict(l=10, r=10, t=10, b=10), showlegend=False,
        xaxis=dict(range=[0, 140], visible=False, fixedrange=True),
        yaxis=dict(range=[0, 120], visible=False, fixedrange=True),
        plot_bgcolor=BG_COLOR, paper_bgcolor=BG_COLOR)

    fig.write_html(OUT / 'dashboard_sketch.html', include_plotlyjs=True, full_html=True)
    return fig

## 6. Run and validate the analysis

Run this cell to build the city-hour, event, intensity, and recovery tables used by the charts and app. The checks reconcile rainfall counts to observed hours and verify estimate eligibility, event-panel coverage, and consistency between recovery labels and times.

In [77]:
hours, events, intensity, recovery = main()

assert not hours.duplicated(['city', 'timestamp']).any()
assert int(coverage.sum().sum()) == len(hours)
supported = hours.supported_sensor_rows.ge(MIN_SENSORS) & hours.supported_sensor_share.ge(MIN_PANEL_SHARE)
assert hours.median_flow_change_pct.notna().eq(supported).all()
assert intensity.median_change_pct.notna().eq(intensity.display_supported).all()

assert not events.event_id.duplicated().any()
assert events.peak_rain_mm.ge(EVENT_PEAK_MIN_MM).all()
assert events.panel_sensors.ge(MIN_SENSORS).all() and events.panel_share.ge(MIN_PANEL_SHARE).all()
assert events.followup_hours.between(MIN_POST_HOURS, MAX_POST_HOURS).all()
deficit = events.flow_deficit_observed.astype(bool)
confirmed = events.recovery_confirmed_hours.notna()
assert (~confirmed | deficit).all()
assert events.loc[confirmed, 'recovery_confirmed_hours'].between(2, events.loc[confirmed, 'followup_hours']).all()
assert events.recovery_status.eq('no measured deficit').eq(~deficit).all()
assert events.recovery_status.eq('confirmed within follow-up').eq(confirmed).all()
assert events.recovery_status.eq('not confirmed within follow-up').eq(deficit & ~confirmed).all()
assert int(recovery.well_observed_events.sum()) == len(events)

print('Analysis and validation complete | Source:', DATA_SOURCE,
      '| city-hours:', len(hours), '| events:', len(events))


Processing city: paris...
Processing city: luzern...
Processing city: hamburg...
Processing city: toronto...
Processing city: essen...
Processing city: innsbruck...
Processing city: darmstadt...
Processing city: manchester...
Generating and saving interactive Plotly charts...
Creating dashboard mockup layout sketch...
Analysis and validation complete | Source: google_cloud | city-hours: 23799 | events: 118


## 7. Milestone 1 · Interactive Plotly charts

Use the three figures below to compare rainfall coverage, traffic-flow deviations by intensity, and flow around the end of rain. Run the chart cells after Section 6 to regenerate them from the selected source and enable hover and zoom in Colab.

### Rainfall coverage

Bar length shows the observed city-hours in each rainfall class. Start with **Hours** to compare coverage, then choose **Percentage** to compare rainfall composition within each city. Hover displays the count and share. A hatch identifies explicitly missing precipitation. The dry class can include zeros imputed before export.

In [78]:
coverage_fig.show()

### Traffic flow by rainfall intensity

The heatmap shows the median of supported city-hour deviations for each city and rainfall class. A value requires at least 10 supported hours across at least three dates. The symmetric color scale centers on zero, and the dry column provides a reference for the rainy cells.

- **NR** means there were no observed hours in the class.
- **NB** means the class had observations but lacked baseline-supported city-hours.
- **SP** means the class had too few supported hours or dates.
- **0.0%** means the estimate is zero at the displayed precision.

Hover for observed hours, supported hours, and supported dates. The summary identifies the largest absolute deviation among rainy classes. Arrows mark Innsbruck’s dry reference and Luzern’s heavy-rain estimate when those cells have sufficient data.

In [79]:
intensity_fig.show()

### Flow around the end of rain

Each city panel aligns qualifying events at the start of the first dry hour, marked as hour 0. The blue line shows the median deviation across events. The blue band spans the 25th to 75th percentiles of the observed events, and the gray band marks the ±10% recovery window. Hover to see the event count at each time offset.

The curves include all qualifying events. Recovery counts use the subset with a measured deficit. Later points can include fewer events as follow-up ends.

In [80]:
recovery_fig.show()
print('Measured deficits:', int(recovery.events_with_deficit.sum()),
      '| Confirmed recoveries:', int(recovery.confirmed_recoveries.sum()),
      '| Unconfirmed:', int(recovery.not_confirmed_in_window.sum()))


Measured deficits: 41 | Confirmed recoveries: 23 | Unconfirmed: 18


## 7a. Additional rainfall and traffic plots

These views use the same city-hour and event tables. Flow deviations are percentages relative to matched dry baselines. Run the individual cells to use their menus and hover details.

In [81]:
supported_hours = prepare_supported_hours(hours)


### Traffic patterns by hour of day

Compare median deviations and the number of supported dry and rainy city-hours at each recorded clock hour. Each city-hour has equal weight in **All cities**. Use the city menu for a within-city comparison. Rainy hours begin at 0.1 mm/hour.

In [82]:
fig_hourly = make_hourly_traffic_chart(supported_hours)
fig_hourly.show()


### Rainfall intensity and flow deviation

Each point represents one supported rainy city-hour. Choose a city and hover for the timestamp, rainfall, deviation, and number of supported sensors. The initial vertical range is −105% to +100%. Choose **Full range** to see larger deviations.

In [83]:
fig_rain_scatter = make_rain_flow_scatter(supported_hours)
fig_rain_scatter.show()


### Daily rainfall calendar

Pick a city and year to see precipitation summed across observed hours each day. An **×** marks a partial day with 1–23 recorded hourly amounts. Gray marks a day within the observation span with no recorded rainfall amount. Blank cells fall outside the observation span or the selected year.

In [84]:
fig_calendar = make_rainfall_calendar(hours)
fig_calendar.show()


### Recovery outcomes by city

Each qualifying event contributes to one outcome. Compare how many events had no measured deficit, how many had a confirmed recovery, and how many remained unconfirmed at the end of follow-up.

In [85]:
fig_event_outcomes = make_event_outcomes_chart(events)
fig_event_outcomes.show()


### Distribution of flow deviations

Each box shows the median and middle half of supported city-hour deviations for one rainfall class. Whiskers extend to observations within 1.5 times the interquartile range, and dots show observations beyond the whiskers. Counts show how many city-hours contribute to each class. Choose **Full range** to see deviations outside the initial −105% to +100% view.

In [86]:
fig_flow_boxes = make_flow_distribution_chart(supported_hours)
fig_flow_boxes.show()


### Event duration and recovery follow-up

The plot includes qualifying events with a measured flow deficit. Filled circles show the confirmation time. Open triangles show the last observed hour for events without a confirmed recovery, so their vertical position measures follow-up length. Horizontal offsets separate overlapping points, and hover reports exact event durations.

In [87]:
fig_event_duration = make_event_duration_chart(events)
fig_event_duration.show()


## 8. Recovery results

Read each city's median confirmation time alongside the numbers of events with a measured deficit, confirmed recoveries, and unconfirmed recoveries. The median uses confirmed recoveries only. Unequal follow-up limits comparisons across cities.

In [88]:
recovery.round(2)

,well_observed_events,events_with_deficit,confirmed_recoveries,not_confirmed_in_window,median_recovery_hours
city,,,,,
paris,56,16,8,8,2.0
luzern,28,15,8,7,2.0
hamburg,13,5,4,1,4.5
toronto,6,1,1,0,3.0
essen,4,1,0,1,NaN
innsbruck,3,1,1,0,6.0
darmstadt,4,1,0,1,NaN
manchester,4,1,1,0,6.0


## 9. Dashboard sketch

The sketch shows the layout of the five dashboard views. The analysis writes `dashboard_sketch.html` to the selected output directory.

In [89]:
dashboard_fig.show(config={'displayModeBar':False})

## 10. Milestone 2 · Interactive Dash app

Run **My Web App** after completing Section 6, then run **Open the dashboard**. In Colab the dashboard appears directly below the cell. Keep the runtime connected while using it. In local Jupyter, click the localhost link printed by the cell.

- **Cities** applies to all five charts.
- **Day type** uses the recorded date for hourly charts and the event start date for event charts.
- **Rainfall intensity** uses each hour's rainfall for coverage and the heatmap, and each event's peak rainfall for the storm scatter and recovery curves. The hour-of-day profile keeps both dry and rainy hours visible.
- Click a point in the storm scatter to highlight that event in the recovery curves.

### Color-blind accessibility verification

The dashboard uses named rainfall classes, direct tooltips, text labels, and rainfall patterns, so color is not the only way to interpret a result. Rainfall colors were changed to distinct blue, orange, yellow, and dark-indigo hues, while the zero-centered flow scale uses blue for positive deviations, neutral white near zero, and brown/orange for negative deviations. Unknown rainfall is additionally marked with a hatch.

The palette cell runs a reproducible approximate simulation for protanopia, deuteranopia, and tritanopia. **The simulation and corrections are complete:** all three cases passed the documented minimum pairwise separation threshold of 35 RGB units. The check raises an error if a future palette revision fails, and the chart legends, axis labels, hover text, and city markers remain available even under reduced color discrimination.

In [90]:
# @title My Web App
# This is the only application cell. It uses tables computed from the Google Cloud CSV above.
DASH_THEME = {
    'page': '#F4F4EB', 'surface': '#FFFDF6', 'sage': '#365A49',
    'sage_light': '#E4ECE4', 'sage_border': '#C8D6C9', 'ink': '#26372D',
    'muted': '#53645A', 'insight': '#E8F0E7', 'accent': '#52745C'
}
if not {'hours', 'events', 'curves'} <= globals().keys():
    raise RuntimeError('Run through Section 6 before starting My Web App.')

app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

hours_df, events_df, curves_df = hours.copy(), events.copy(), curves.copy()
hours_df['timestamp'] = pd.to_datetime(hours_df.timestamp)
events_df['start'] = pd.to_datetime(events_df.start)
INTENSITY_COLOR_LIMIT = intensity_color_limit(summarize_intensity(hours_df))


def kpi_card(label, value_id, detail_id=None, accent=None):
    accent = accent or DASH_THEME['sage']
    content = [
        html.P(label, className="mb-2", style={
            'color': DASH_THEME['muted'], 'fontSize': '0.82rem', 'fontWeight': '700',
            'letterSpacing': '0.06em', 'textTransform': 'uppercase'
        }),
        html.H2(id=value_id, className="mb-0", style={
            'color': DASH_THEME['ink'], 'fontSize': 'clamp(1.7rem, 3vw, 2.2rem)',
            'fontWeight': '750', 'lineHeight': '1.1'
        })
    ]
    if detail_id:
        content.append(html.P(id=detail_id, className="mt-2 mb-0", style={
            'color': DASH_THEME['muted'], 'fontSize': '0.85rem', 'lineHeight': '1.35'
        }))
    return dbc.Card(dbc.CardBody(content, className="p-3 p-lg-4"), style={
        'backgroundColor': DASH_THEME['surface'], 'border': f"1px solid {DASH_THEME['sage_border']}",
        'borderTop': f'4px solid {accent}', 'borderRadius': '14px',
        'boxShadow': '0 3px 12px rgba(38,55,45,.07)', 'height': '100%'
    })


# Give Plotly a stable container to resize within when filters redraw the figure.
DASH_GRAPH_HEIGHTS = (520, 576, 560, 500, 700)


def chart_section(number, title, description, insight_id, graph_id):
    insight = [html.Div(id=insight_id, className="mb-3 p-3", role="status", style={
        'backgroundColor': DASH_THEME['insight'], 'borderLeft': f"4px solid {DASH_THEME['accent']}",
        'borderRadius': '6px', 'color': DASH_THEME['ink'], 'lineHeight': '1.5'
    })] if insight_id else []
    return dbc.Card(dbc.CardBody([
        html.H2(f'{number}. {title}', className="h4 mb-2", style={'color': DASH_THEME['sage'], 'fontWeight': '700'}),
        html.P(description, className="mb-3", style={'color': DASH_THEME['muted'], 'lineHeight': '1.55'}),
        *insight,
        dcc.Loading(type='circle', color=DASH_THEME['sage'], delay_show=250,
        children=dcc.Graph(id=graph_id, responsive=True,
            config={'displaylogo': False, 'scrollZoom': True},
            style={'height': f'min(70vh, {DASH_GRAPH_HEIGHTS[int(number) - 1]}px)', 'minHeight': '360px'}))
    ], className="p-3 p-lg-4"), style={
        'backgroundColor': DASH_THEME['surface'], 'border': f"1px solid {DASH_THEME['sage_border']}",
        'borderRadius': '14px', 'boxShadow': '0 3px 12px rgba(38,55,45,.06)'
    })


def configure_dashboard_figure(figure):
    figure.update_layout(hovermode='closest',
        legend=dict(itemclick='toggle', itemdoubleclick='toggleothers'))
    figure.update_xaxes(showspikes=True, spikemode='across', spikesnap='cursor',
        spikecolor='#52606D', spikethickness=1)
    figure.update_yaxes(showspikes=True, spikemode='across', spikesnap='cursor',
        spikecolor='#52606D', spikethickness=1)
    return figure


app.layout = dbc.Container([
    dbc.Row([dbc.Col(html.Header([
        html.P('IUTF | RAINFALL + URBAN TRAFFIC', className="mb-2", style={
            'color': '#E5EEE5', 'fontSize': '0.78rem', 'fontWeight': '700', 'letterSpacing': '0.12em'
        }),
        html.H1('Rainfall & Traffic Flow Across Eight Cities', className="display-5 mb-3", style={
            'color': '#FFFDF6', 'fontWeight': '750', 'lineHeight': '1.12'
        }),
        html.P('Five views of rainfall, traffic flow, and event recovery across eight cities.',
            className="lead mb-3", style={'color': '#F1F4EC', 'maxWidth': '940px', 'fontSize': '1.08rem'}),
        html.P('Cloud weather matches may be incomplete. Results describe observed associations.',
            className="mb-0", style={'color': '#E5EEE5', 'fontSize': '0.92rem', 'maxWidth': '1000px'})
    ], className="p-4 p-lg-5", style={
        'backgroundColor': DASH_THEME['sage'], 'borderRadius': '18px',
        'boxShadow': '0 5px 18px rgba(38,55,45,.14)'
    }), width=12)], className="pt-4 pb-3"),

    dbc.Card(dbc.CardBody([
        html.H2('Filters & Controls', className="h5 mb-3", style={'color': DASH_THEME['sage'], 'fontWeight': '700'}),
        dbc.Row([
            dbc.Col([html.Label('Cities', htmlFor="city-dropdown", style={'fontWeight': '700', 'color': DASH_THEME['ink']}),
                dcc.Dropdown(id="city-dropdown", options=[{'label': NAMES[c], 'value': c} for c in CITIES],
                    value=CITIES, multi=True, placeholder='Select one or more cities...')], xs=12, md=4),
            dbc.Col([html.Label('Day type', htmlFor="day-type-dropdown", style={'fontWeight': '700', 'color': DASH_THEME['ink']}),
                dcc.Dropdown(id="day-type-dropdown", options=[
                    {'label': 'All days', 'value': 'all'}, {'label': 'Weekdays', 'value': 'weekday'},
                    {'label': 'Weekends', 'value': 'weekend'}], value='all', clearable=False)], xs=12, md=4),
            dbc.Col([html.Label('Rainfall intensity', htmlFor="intensity-dropdown", style={'fontWeight': '700', 'color': DASH_THEME['ink']}),
                dcc.Dropdown(id="intensity-dropdown", options=[{'label': 'All intensities', 'value': 'all'}] +
                    [{'label': label, 'value': label} for label in LABELS], value='all', clearable=False)], xs=12, md=4)
        ], className="g-3")
    ], className="p-3 p-lg-4"), className="mb-4", style={
        'backgroundColor': DASH_THEME['surface'], 'border': f"1px solid {DASH_THEME['sage_border']}",
        'borderRadius': '14px'
    }),

    dcc.Markdown('''Why these five views: the dashboard follows one decision path, while the **additional plots stay in Section 7a**. The five views cover (1) where rainfall is observed, (2) how flow changes by rainfall intensity, (3) how flow varies across the day, (4) whether storm intensity relates to disruption, and (5) whether flow returns after rain ends. The additional hourly, individual-hour scatter, daily-calendar, outcome-count, distribution, and event-duration plots provide deeper or narrower exploration, but they repeat or split the five main questions. Keeping them separate makes the Dash app easier to scan while retaining them as supporting evidence.''', className="mb-4 px-3 py-3", style={
        'backgroundColor': DASH_THEME['insight'], 'borderLeft': f"4px solid {DASH_THEME['accent']}",
        'borderRadius': '10px', 'color': DASH_THEME['ink'], 'lineHeight': '1.55'
    }),
    dbc.Row([
        dbc.Col(kpi_card('Selected cities','kpi-cities',accent='#52745C'), xs=12, sm=6, lg=3),
        dbc.Col(kpi_card('Observed city-hours','kpi-hours',accent='#47758A'), xs=12, sm=6, lg=3),
        dbc.Col(kpi_card('Well-observed rain events','kpi-events',accent='#8A6A39'), xs=12, sm=6, lg=3),
        dbc.Col(kpi_card('Confirmed recoveries','kpi-recovery','kpi-recovery-detail',accent='#6A5D80'), xs=12, sm=6, lg=3)
    ], className="g-3 mb-4"),

    dbc.Row([
        dbc.Col(chart_section('1','Rainfall coverage',
            'Stacked city-hours by rainfall class. Toggle hours or within-city percentages.',
            'coverage-insight','live-coverage-chart'), xs=12, lg=5),
        dbc.Col(chart_section('2','Traffic flow by rainfall intensity',
            'Median flow deviation from matched dry baselines by rainfall intensity.',
            'intensity-insight','live-intensity-chart'), xs=12, lg=7)
    ], className="g-3 mb-4"),

    dbc.Row([
        dbc.Col(chart_section(str(number), title, description, None, f'live-{key}-chart'), xs=12, md=6)
        for number, (key, title, description) in enumerate(DASH_COMPARISON_CHARTS, start=3)
    ], className="g-3 mb-4"),

    dbc.Row([dbc.Col(chart_section('5','Traffic flow after rain ends',
        'Events are aligned at the first dry hour (hour 0). The line is the event median; the shaded band is the middle 50% of events.',
        'recovery-insight','live-recovery-chart'), width=12)], className="mb-4"),

    html.Footer([
        'Data source · ',
        html.A('Google Cloud Storage CSV', href=DATA_URL, target='_blank', rel='noopener noreferrer')
        if DATA_SOURCE == 'google_cloud' else 'IUTFD raw Parquet files'
    ], className="text-center py-3", style={'color': DASH_THEME['muted'], 'fontSize': '0.85rem'})
], fluid=True, className="px-3 px-lg-4 pb-4", style={
    'backgroundColor': DASH_THEME['page'], 'color': DASH_THEME['ink'],
    'fontFamily': 'Inter, Segoe UI, Arial, sans-serif', 'minHeight': '100vh',
    'maxWidth': '1680px', 'margin': '0 auto'
})

@app.callback(
    [Output("kpi-cities", "children"), Output("kpi-hours", "children"), Output("kpi-events", "children"),
     Output("kpi-recovery", "children"), Output("kpi-recovery-detail", "children"),
     Output("coverage-insight", "children"), Output("intensity-insight", "children"), Output("recovery-insight", "children"),
     Output("live-coverage-chart", "figure"), Output("live-intensity-chart", "figure"),
     *[Output(f'live-{key}-chart', 'figure') for key, _, _ in DASH_COMPARISON_CHARTS],
     Output("live-recovery-chart", "figure"), Output("live-recovery-chart", "style")],
    [Input("city-dropdown", "value"), Input("day-type-dropdown", "value"),
     Input("intensity-dropdown", "value"), Input("live-storm-disruption-chart", "clickData")]
)
def update_dashboard(selected_cities, day_type, intensity_filter, event_click_data=None):
    selected_cities = [city for city in (selected_cities or []) if city in CITIES]
    if not selected_cities:
        def empty_figure(title):
            figure = go.Figure()
            figure.add_annotation(text="Select at least one city to display this chart.", showarrow=False,
                                  x=.5, y=.5, xref="paper", yref="paper")
            figure.update_layout(title=title, template="plotly_white", paper_bgcolor=DASH_THEME['surface'])
            return figure
        message = "Select one or more cities to see data-driven insights."
        return ("0", "0", "0", "0 / 0", "Select cities to see recovery counts.", message, message, message,
                empty_figure("Rainfall coverage"), empty_figure("Traffic flow by rainfall intensity"),
                *[empty_figure(title) for _, title, _ in DASH_COMPARISON_CHARTS],
                empty_figure("Recovery curves"), {'height': 'min(70vh, 700px)', 'minHeight': '360px'})

    filtered_hours = hours_df.loc[hours_df.city.isin(selected_cities)]
    filtered_events = events_df[events_df.city.isin(selected_cities)].copy()
    filtered_curves = curves_df[curves_df.city.isin(selected_cities)].copy()
    if day_type == 'weekday':
        filtered_hours = filtered_hours[filtered_hours.timestamp.dt.dayofweek < 5]
    elif day_type == 'weekend':
        filtered_hours = filtered_hours[filtered_hours.timestamp.dt.dayofweek >= 5]

    if day_type == 'weekday':
        filtered_events = filtered_events[filtered_events.start.dt.dayofweek < 5]
    elif day_type == 'weekend':
        filtered_events = filtered_events[filtered_events.start.dt.dayofweek >= 5]

    comparison_hours = filtered_hours
    if intensity_filter != 'all':
        filtered_hours = filtered_hours[filtered_hours.rain_bin.astype('string').eq(intensity_filter)]
        peak_bins = pd.cut(pd.to_numeric(filtered_events.peak_rain_mm, errors='coerce'),
                           bins=BINS, labels=LABELS, right=False)
        filtered_events = filtered_events[peak_bins.astype('string').eq(intensity_filter)]

    if 'event_id' in filtered_events and 'event_id' in filtered_curves:
        filtered_curves = filtered_curves[filtered_curves.event_id.isin(filtered_events.event_id)]
    else:
        filtered_curves = filtered_curves.iloc[0:0]
    selected_event = None
    if event_click_data and 'event_id' in filtered_events:
        for point in event_click_data.get('points', []):
            clicked_event_id = point.get('customdata')
            if isinstance(clicked_event_id, (list, tuple)):
                clicked_event_id = clicked_event_id[0] if clicked_event_id else None
            if clicked_event_id is not None:
                match = filtered_events.loc[
                    filtered_events.event_id.astype(str).eq(str(clicked_event_id))
                ]
                if not match.empty:
                    selected_event = match.iloc[0].to_dict()
                    break

    coverage = filtered_hours.groupby(['city','rain_bin'], observed=False).size().unstack(fill_value=0).reindex(
        index=selected_cities, columns=LABELS, fill_value=0)
    coverage['Unknown'] = filtered_hours[filtered_hours.rain_bin.isna()].groupby('city').size().reindex(
        selected_cities, fill_value=0)
    intensity = summarize_intensity(filtered_hours)
    curve_summary = filtered_curves.groupby(['city','hours_from_rain_end']).flow_change_pct.agg(
        median='median', q25=lambda values: values.quantile(.25),
        q75=lambda values: values.quantile(.75), events='count').reset_index()
    recovery = recovery_statistics(filtered_events, selected_cities)
    deficits = int(recovery.events_with_deficit.sum())
    confirmed = int(recovery.confirmed_recoveries.sum())
    event_label = 'event' if deficits == 1 else 'events'
    recovery_detail = f'{deficits:,} {event_label} had a measured flow deficit of at least 10%.'

    comparison = prepare_supported_hours(comparison_hours)
    figures = (
        make_coverage_chart(coverage, selected_cities, show_insight=False),
        make_intensity_chart(intensity, selected_cities, color_limit=INTENSITY_COLOR_LIMIT, show_insight=False),
        make_hourly_traffic_chart(comparison, city_menu=False),
        make_storm_disruption_chart(filtered_events, selected_cities),
        make_recovery_chart(curve_summary, recovery, selected_cities, show_insight=False,
                            selected_event=selected_event, event_curves=filtered_curves),
    )
    figures = tuple(configure_dashboard_figure(figure) for figure in figures)
    recovery_rows = max(1, (len(selected_cities) - 1) // 2 + 1)
    recovery_graph_style = {
        'height': f'max(70vh, {350 * recovery_rows}px)', 'minHeight': '360px'
    }
    insights = (coverage_insight(coverage), intensity_insight(intensity), recovery_insight(recovery))
    return (str(len(selected_cities)), f"{len(filtered_hours):,}", str(len(filtered_events)),
            f'{confirmed:,} / {deficits:,}', recovery_detail, *insights, *figures,
            recovery_graph_style)


from werkzeug.serving import make_server


previous_server = globals().get('dash_server')
previous_thread = globals().get('dash_thread')
if previous_server is not None:
    if previous_thread is not None and previous_thread.is_alive():
        previous_server.shutdown()
    previous_server.server_close()

dash_server = make_server('127.0.0.1', 0, app.server, threaded=True)
dash_thread = threading.Thread(target=dash_server.serve_forever, daemon=True)
dash_thread.start()
print('Dash is running. Run Open the dashboard to view it.')


Dash is running. Run Open the dashboard to view it.


In [91]:
# @title Open the dashboard
from IPython.display import Markdown, display
from urllib.request import urlopen

if globals().get('dash_thread') is None or not dash_thread.is_alive():
    raise RuntimeError('Run My Web App in the current runtime, then rerun this cell.')

dashboard_url = f'http://127.0.0.1:{dash_server.server_port}/'
try:
    with urlopen(dashboard_url + '_dash-layout', timeout=10):
        pass
except OSError as error:
    raise RuntimeError('Dash did not respond. Rerun My Web App, then reopen the dashboard.') from error

try:
    from google.colab import output
except ImportError:
    display(Markdown(f'[Open the Dash dashboard]({dashboard_url})'))
else:
    output.serve_kernel_port_as_iframe(dash_server.server_port, height=900)


127.0.0.1 - - [22/Nov/2026 15:11:22] "GET /_dash-layout HTTP/1.1" 200 -


[Open the Dash dashboard](http://127.0.0.1:50661/)

## 11. Limitations and further checks

- **Weather alignment.** The cloud export has unresolved timestamp and weather-match limitations. Coarse ERA5 grids and hourly aggregation can smooth local or short bursts of rain.
- **Coverage.** Cities contribute different years, seasons, and numbers of observed days. The analysis leaves season and holidays unadjusted. The deepest Paris dip begins at midnight on January 2, 2016, overnight after New Year’s Day. The overnight timing is consistent with post-holiday travel patterns, but the data cannot identify holiday effects; this is only a possible explanation. Heavy and extreme rain can have sparse coverage. [French public holidays in 2016](https://www.legifrance.gouv.fr/codes/section_lc/LEGITEXT000006072050/LEGISCTA000006178007/2016-02-29/)
- **Traffic sensor quality.** A recorded zero can reflect low traffic or a sensor issue. Upstream flags and sampling completeness remain partly unaudited.
- **Recovery.** Confirmation depends on sensor-panel eligibility, the ±10% tolerance, and the available follow-up.

A corrected weather export would make it possible to check alignment directly. Sensitivity checks could then compare the results across alternative dry thresholds, support requirements, recovery tolerances, and seasons.

## 12. Findings from the selected source

The output below reports observed city-hours, qualifying events, recovery outcomes, and intensity estimates from this run. Compare rainy cells with the dry column and check how many hours and dates support each estimate. Blank values indicate that the estimate did not meet the support rule.

Cloud results describe observed associations under the supplied weather join. Read the recovery counts together with the follow-up rules in Section 4.

In [92]:
print('Current analysis source:', DATA_SOURCE)
print('Observed city-hours:', len(hours), '| qualifying events:', len(events))
print('Measured deficits:', int(recovery.events_with_deficit.sum()),
      '| confirmed recoveries:', int(recovery.confirmed_recoveries.sum()),
      '| not confirmed:', int(recovery.not_confirmed_in_window.sum()))
display(intensity[['city','rain_bin','median_change_pct','city_hours','supported_dates','observed_city_hours','display_supported']])
if DATA_SOURCE == 'google_cloud':
    print('Provisional: the merged CSV does not allow full weather-join repair or identification of imputed zeros.')


Current analysis source: google_cloud
Observed city-hours: 23799 | qualifying events: 118
Measured deficits: 41 | confirmed recoveries: 23 | not confirmed: 18


,city,rain_bin,median_change_pct,city_hours,supported_dates,observed_city_hours,display_supported
0,darmstadt,Dry,-2.489168,779,39,779,True
1,darmstadt,Light,0.269087,82,16,82,True
2,darmstadt,Moderate,-1.609294,39,10,39,True
3,essen,Dry,-0.953904,630,35,630,True
4,essen,Light,-1.780941,135,23,135,True
5,essen,Moderate,1.295591,58,11,58,True
6,essen,Heavy,NaN,1,1,1,False
7,hamburg,Dry,-0.603890,2237,105,2237,True
8,hamburg,Light,-1.463319,214,48,214,True
9,hamburg,Moderate,-2.196353,57,20,57,True


Provisional: the merged CSV does not allow full weather-join repair or identification of imputed zeros.


## Walkthrough and submission

**Stream recording link pending.** Record a walkthrough of no more than eight minutes in Stream using your IU account and include the camera view. Share with **Anyone** or **People in Indiana University**, as the assignment permits.


## Appendix · Reproduce the historical audit

The historical tables above use the embedded audit snapshot. To rerun the historical audit, place the full `IUTFD/` collection, `all_cities_combined.csv`, and `iutf_project.csv` under `ROOT`. Then set `RUN_FULL_CITY_AUDIT = True` in the cell below the audit function.

In [93]:
# @title Optional: full-city raw-data audit (disabled)
def run_full_city_audit(root):
    """Optional: recompute the original 40-folder local eligibility audit."""
    from pathlib import Path
    import json
    import pandas as pd

    ROOT = Path(root)
    OUT = ROOT / 'city_selection'


    def profile_city(path):
        city = path.name
        f = path / 'sensors/hourly_readings.parquet'
        if not f.exists():
            return {'city': city, 'status': 'no hourly readings'}
        d = pd.read_parquet(f)
        d['timestamp'] = pd.to_datetime(d['datetime'], format='mixed', dayfirst=True)
        d['detid'] = d.detid.astype(str)
        ts = pd.DatetimeIndex(d.timestamp.unique()).sort_values()
        r = {'city': city, 'status': 'profiled', 'rows': len(d),
             'sensors': d.detid.nunique(), 'observed_days': len(set(ts.date)),
             'observed_hours': len(ts), 'start': str(ts.min()), 'end': str(ts.max()),
             'span_days': (ts.max().normalize()-ts.min().normalize()).days+1,
             'duplicate_sensor_hours': int(d.duplicated(['detid','timestamp']).sum()),
             'speed_available_pct': 100*d.speed_mean.notna().mean(),
             'flow_missing_pct': 100*d.flow_mean_5min.isna().mean(),
             'negative_flow_rows': int((d.flow_mean_5min<0).sum()),
             'zero_flow_pct': 100*d.flow_mean_5min.eq(0).mean()}
        meta_path = path / 'sensors/detectors_info.parquet'
        if meta_path.exists():
            m = pd.read_parquet(meta_path); m['detid'] = m.detid.astype(str)
            r['duplicate_detector_ids'] = int(m.detid.duplicated().sum())
            r['unmatched_detector_pct'] = 100*(~d.detid.isin(m.detid)).mean()
            coords = m.set_index('detid')[['lat','long']]
            if coords.index.is_unique:
                r['coordinates_missing_pct'] = 100*d.detid.map(coords.notna().all(axis=1)).fillna(False).eq(False).mean()
        files = sorted((path/'weather/datetime').glob('*.parquet'))
        if not files:
            r['status'] = 'no weather files'; return r
        w = pd.concat([pd.read_parquet(p, columns=['utc_time','local_time','grid_id','total_precipitation']) for p in files], ignore_index=True)
        w['local_time'] = pd.to_datetime(w.local_time, format='mixed')
        w['utc_naive'] = pd.to_datetime(w.utc_time, utc=True).dt.tz_localize(None)
        r['duplicate_weather_grid_hours'] = int(w.duplicated(['utc_time','grid_id']).sum())
        r['negative_rain_rows'] = int((w.total_precipitation<0).sum())
        local = w.groupby('local_time').total_precipitation.mean()
        utc = w.groupby('utc_naive').total_precipitation.mean()
        h = pd.DataFrame(index=ts)
        h['rain_mm'] = local.reindex(ts)*1000
        h['old_utc_rain_mm'] = utc.reindex(ts)*1000
        r['weather_matched_pct'] = 100*h.rain_mm.notna().mean()
        r['old_utc_unmatched_pct'] = 100*h.old_utc_rain_mm.isna().mean()
        both = h.dropna()
        r['rain_changed_by_time_alignment_hours'] = int(((both.rain_mm-both.old_utc_rain_mm).abs().gt(1e-6 + 1e-5*both.old_utc_rain_mm.abs())).sum())
        r['moderate_hours'] = int(((h.rain_mm>=0.5)&(h.rain_mm<4)).sum())
        r['heavy_hours'] = int(((h.rain_mm>=4)&(h.rain_mm<10)).sum())
        r['extreme_hours'] = int((h.rain_mm>=10).sum())
        r['wet_hours'] = int((h.rain_mm>=0.5).sum())
        r['dry_light_hours'] = int((h.rain_mm<0.5).sum())
        r['max_rain_mm'] = h.rain_mm.max()
        # Events require an observed dry/light preceding hour; gaps do not start events.
        wet = h.rain_mm.ge(0.5)
        onsets = ts[wet & h.rain_mm.shift().lt(0.5) & ts.to_series(index=ts).diff().eq(pd.Timedelta(hours=1))]
        r['observed_wet_onsets'] = len(onsets)
        # Strict recovery screen: 3 dry hours before onset, complete event, then
        # 3 dry hours after its end. Counts are not independent causal replicates.
        recovery = 0
        for t in onsets:
            pre = h.rain_mm.reindex(pd.date_range(t-pd.Timedelta(hours=3), periods=3, freq='h'))
            if not (pre.notna().all() and pre.lt(0.5).all()): continue
            end = t
            while end in h.index and pd.notna(h.at[end,'rain_mm']) and h.at[end,'rain_mm']>=0.5:
                end += pd.Timedelta(hours=1)
            post = h.rain_mm.reindex(pd.date_range(end, periods=3, freq='h'))
            if post.notna().all() and post.lt(0.5).all(): recovery += 1
        r['complete_recovery_windows'] = recovery
        wet_d = d[d.timestamp.isin(ts[wet])]
        r['wet_sensor_rows'] = len(wet_d)
        r['wet_speed_available_pct'] = 100*wet_d.speed_mean.notna().mean() if len(wet_d) else float('nan')
        matched = d.timestamp.map(local)*1000
        r['wet_baseline_missing_pct'] = float('nan')
        r['wet_baseline_under3_pct'] = float('nan')
        if len(wet_d):
            keys = pd.DataFrame({'detid':d.detid, 'hour':d.timestamp.dt.hour, 'weekday':d.timestamp.dt.dayofweek})
            counts = keys[matched.lt(0.5) & d.flow_mean_5min.notna()].value_counts()
            wk = pd.MultiIndex.from_frame(keys.loc[wet_d.index])
            support = counts.reindex(wk).fillna(0)
            r['wet_baseline_missing_pct'] = 100*support.eq(0).mean()
            r['wet_baseline_under3_pct'] = 100*support.lt(3).mean()
        h.reset_index(names='local_timestamp').to_csv(OUT/f'{city}_hourly_weather_audit.csv',index=False)
        return r


    def scan_csv(name):
        acc = {}
        columns = ['city','timestamp','speed','flow','precipitation','rain_category','baseline_flow','pct_flow_change','lat','lon']
        for chunk in pd.read_csv(ROOT/name,usecols=columns,chunksize=250000):
            for city,g in chunk.groupby('city'):
                a=acc.setdefault(city, {'city':city,'rows':0,'speed_missing':0,'flow_missing':0,'baseline_missing':0,'zero_baseline':0,'infinite_pct_change':0,'coordinates_missing':0,'non_light_category_rows':0,'max_precipitation':0,'timestamps':set()})
                a['rows'] += len(g)
                for col in ['speed','flow']: a[col+'_missing'] += int(g[col].isna().sum())
                a['baseline_missing'] += int(g.baseline_flow.isna().sum())
                a['zero_baseline'] += int(g.baseline_flow.eq(0).sum())
                a['infinite_pct_change'] += int(g.pct_flow_change.isin([float('inf'),float('-inf')]).sum())
                a['coordinates_missing'] += int(g[['lat','lon']].isna().any(axis=1).sum())
                a['non_light_category_rows'] += int(g.rain_category.ne('None/Light').sum())
                a['max_precipitation'] = max(a['max_precipitation'],g.precipitation.max())
                a['timestamps'].update(g.timestamp.unique())
        for a in acc.values():
            ts=a.pop('timestamps'); a['observed_hours']=len(ts); a['observed_days']=len({t[:10] for t in ts})
        result=pd.DataFrame(acc.values()).sort_values('city')
        result.to_csv(OUT/(Path(name).stem+'_profile.csv'),index=False)
        return result


    def main():
        OUT.mkdir(exist_ok=True)
        rows=[]
        for p in sorted((ROOT/'IUTFD').iterdir()):
            if p.is_dir():
                r=profile_city(p); rows.append(r)
                print(p.name,r.get('observed_days'),r.get('wet_hours'),r.get('complete_recovery_windows'),flush=True)
        profile=pd.DataFrame(rows)
        profile.to_csv(OUT/'all_city_quality.csv',index=False)
        for name in ['all_cities_combined.csv','iutf_project.csv']:
            s=scan_csv(name)
            print(name,int(s.rows.sum()),len(s),'non-light rows',int(s.non_light_category_rows.sum()),flush=True)
        return profile



    return main()

In [94]:
RUN_FULL_CITY_AUDIT = False
if RUN_FULL_CITY_AUDIT:
    refreshed_audit = run_full_city_audit(ROOT)
    display(refreshed_audit)
else:
    print('Full audit not rerun; recorded source-audit tables are embedded earlier in this notebook.')

Full audit not rerun; recorded source-audit tables are embedded earlier in this notebook.


## References and reproduction

- Dataset and methodology. [IUTF Figshare dataset, version 1](https://doi.org/10.6084/m9.figshare.30022807.v1) · [Scientific Data paper](https://www.nature.com/articles/s41597-025-06336-3)
- Precipitation units and accumulation intervals. [ECMWF ERA5 documentation](https://confluence.ecmwf.int/spaces/CKB/pages/76414402/ERA5+data+documentation)
- Traffic hour labels and weather time fields. [Hourly traffic aggregation code](https://github.com/viviRG2024/IUTDF_processing/blob/main/util/calculateHourlySensorData.py) · [Weather preparation code](https://github.com/viviRG2024/IUTDF_processing/blob/main/util/processERA5CityData.py)
- Default source and entry point. [Public Google Cloud Storage CSV](https://storage.googleapis.com/bucket-quickstart-iutfdataset/all_cities_combined.csv) · [Open this notebook in Google Colab](https://colab.research.google.com/github/divyakapoor23/IUTF_Viz/blob/main/FinalProjectIUTF.ipynb)

The full local verification used a checksum-matched copy of the public CSV. Re-running the default workflow loads the public object and rebuilds the analysis tables in memory. The app reads those tables and appears in the notebook output for the active Colab runtime.